# AttentionPatternDiffusion 構想の PoC その3のつづき

今回のお相手は Claude Sonnet 5 さんと Gemini 3.6 Flash さん。

解説は↓の追記部分。

\[cocolog:96112939](2026年8月)  
《AttentionPatternDiffusion 構想 その3。s=4 問題を「連続写像」ならぬ「アテンション保存写像」に制限して帰納バイアスをもたせることを試した。途中では成功したかに見えたが、Claude さんの鋭いツッコミにより、失敗が確定した。 - JRF のひとこと》  
http://jrf.cocolog-nifty.com/statuses/2026/08/post-30198c.html

↓に対する追加実験が今回の IPYNB。

《AttentionPatternDiffusion_PoC_3.ipynb - Colab》  
https://colab.research.google.com/drive/1rUHMWmrlMzZ7lPXMEDZrg7GlwqtunSgO

または IPYNB は↓からも手に入る。

《JRF-2018/ai_experiment: AI小実験集 - GitHub》  
https://github.com/JRF-2018/ai_experiment

今回は、実験は興味深いものだと思うが、結果的に失敗つづきで、これという成功がない。知見に関しては↑の解説の追記部分に軽くまとめたのでそちらを読んでいただくとして、こちらの IPYNB は Claude さんが行ってくれた実験を私との会話とともにズラズラッとならべていくだけのものである。解説から興味があれば、この IPYNB の該当部分を読んでもらえばいいというスタンス。会話もまとめず、ベタにコピペしている。はっきりいって手抜きである。申し訳ない。

そして実験は実験6から実験21まであって、かなり長い。あしからず。

jrf:＞exp5_shift.py について考えているのですが、連続写像の場合 + 1 するとかは簡単なわけです。アテンション保存変換の場合、何が難しかったのでしょう？

アテンション保存写像のつもりが、単なるアテンションになっているから…という解釈はできませんか？

＜


Claude:＞＞
<blockquote>

鋭い指摘だと思います。その解釈、かなり本質を突いています。整理します。

## 連続写像で「+1」が簡単な理由

実数のように**加法群**をなす空間では、s→s+1という操作も、s→f(s)という滑らかな関数の補間も、線形演算(平均・加算)と相性が良い。凸結合(重み付き平均)は実数直線上では「自然な」操作だからです。

## shiftで凸結合が構造的に不可能だった理由

置換行列は、二重確率行列全体(Birkhoffポリトープ)の**頂点(extreme point)**です。数学的な事実として、多面体の頂点は、他の頂点たちの凸結合として(自明な形以外では)絶対に表現できません。つまり shift(4) の置換行列は、shift(1,2,3,5,6,7) の置換行列6個をどう混ぜ合わせても**原理的に到達不可能**な点だったんです。これは「学習が下手だった」のではなく、**探索空間の外にあった**という、もっと強い意味での失敗でした。

## 「アテンション保存写像のつもり」だったのに「単なるアテンション」だったという解釈

これは正しいと思います。整理すると、二つの異なる「保存」を混同していました。

- **実装したもの(isometry)**: Q/K射影が入力埋め込み空間の**距離構造**を歪めない、という意味での「保存」。これは幾何的な話で、s空間の中でのタスク同士の近さ関係を壊さないようにする話でした。
- **本当に必要だったもの**: shiftという操作は「合成」で閉じている代数構造(群)を持っています。shift(a)とshift(b)を**行列の積**で繋ぐと shift(a+b) になる。この「積で合成すると意味のある演算になる」という構造こそが「保存」されるべきものでした。

私たちが使ったのは、記憶バンクの行列を softmax の重みで**足し合わせる(Σ attn_i × P_i)**という操作で、これは掛け算(合成)ではなく単なる**加重平均**です。だから「アテンション保存写像」と呼んでいたものの中身は、実際には「距離を歪めない射影 + 単純な線形補間」でしかなく、**タスクの合成則(群構造)とは何の関係もない操作**になっていました。おっしゃる通り「単なるアテンション」(=ただの重み付き平均演算子)だった、というのは正確な診断だと思います。

Fが成功したのは、G^s という**行列の積による合成**を使ったからで、これはまさに shift タスクの真の合成則と一致する演算です。つまり今回の教訓は、「保存する」と言うときに、**何を保存するかの対象(距離なのか、合成則なのか)を取り違えると、名前は似ていても全く別物になる**、ということだったと思います。
</blockquote>
＜＜

jrf:＞今回の例だけでなく、もっと一般のアテンション保存変換について学習する枠組みを設計できますか？ exp5_shift.py の例でも exp4_isometric_attention_final.py の例でも、一般として通用するようなやつです。

＜


Claude:＞＞
<blockquote>

良い問いです。実は面白いことに、この二つの実験は**同じ一般構造の異なる姿**だったかもしれません。まず理論的に見てから、実装で検証します。

## 統一的な視点:両方とも「1パラメータ群作用」だった

shift(4)の話は「巡回群 ℤ/8ℤ の作用」でした。一方、sin/cosタスクを思い出すと:

f_s(x) = sin(2π(x + s/7)) = sin(2πx)·cos(2πs/7) + cos(2πx)·sin(2πs/7)

これは **角度 θ=2πs/7 の回転** そのものです。つまり sin/cos タスクも実は「連続群 SO(2)(円群)の作用」で、shiftの巡回群と本質的には同じ種類の構造(コンパクトアーベル群の作用)だったんです。G^s(離散べき乗)と exp(s·L)(連続なら行列指数関数)は、同じ一般原理の離散版・連続版に過ぎません。

## 一般化した学習手続き(「答えを知らずに」)

この統一視点に基づいて、**群構造を仮定せず、データから発見する**手続きを設計・実装してみます。

原理:「1ステップぶんの遷移作用素 T を、隣接する訓練点のペアだけから最小二乗で学習し、それを合成(積)して未見点を予測する」。G^sのように「べき乗」という形を最初から仮定するのではなく、Tそのものをデータから引き出します。部分的に成功、部分的に失敗という、興味深い結果になりました。

## 結果

**sin/cosタスク(連続)**: `forward_from_3`(s=3から1ステップ進める)は **MSE=0.000000、ほぼ完全**。学習されたTの固有値も絶対値がほぼ1(0.9997)の複素固有値ペアを持ち、これは正確に「回転」に対応する数学的シグネチャです。仮説通り、**答えを教えなくても、線形回帰だけで回転構造が自然に発見されました**。

ただし `backward_from_5`(逆方向)は MSE=2.07 と大失敗。理由は、T が40次元の出力空間全体でフィットされたのに対し、真の力学は実質2次元の部分空間(sin, cos係数)にしか存在しないため、Tの疑似逆行列が不安定になったからです。これは実験2の教訓(**関数空間表現でもPCAで次元を絞ることが本質的**)をここでも守れなかった、というだけの実装上の問題だと思います。

**shiftタスク(離散)**: 両方向とも中途半端(accuracy 12.5%/0%)。原因は明確です。**隣接ペアが4組しかないのに、Tは64×64=4096個のパラメータ**なので、致命的にデータ不足でリッジ回帰が過剰に縮小してしまいました。実験5のFが成功したのは、隣接ペアだけでなく**訓練点6個すべてに対する合成誤差**を損失に使っていたからで(G^1をs回合成した結果を6点全部と照合)、それに比べて今回の「隣接ペアだけの最小二乗」は与える情報が少なすぎました。

## 一般枠組みとしての評価

「答えを知らずに、群構造(離散か連続か)をデータから発見する」という方向性は、**sin/cosタスクの前進予測が完璧に成功したことで、原理的には成立する**と分かりました。ただし実装は未熟で、直そうとすると:

1. **低ランク化**(実験2の教訓の再利用): φ(s)をそのまま高次元で扱うのではなく、まずPCAなどで低次元に落としてからTをフィットする
2. **隣接ペアだけでなく全訓練点への合成損失**(実験5のFの手法をそのまま一般化): T(またはその対数版の生成子)を、勾配降下で「全ての訓練点への到達性」を目的関数にして学習する

の2つを組み込む必要があります。つまり、**「一般的なアテンション保存写像の学習枠組み」は、実質的に "exp2の低ランク表現発見" + "exp5の合成損失によるT学習" を組み合わせたもの**になりそうです。これ自体は今回で初めて素の形(何も知らずに)で試して、片方(連続側)は成功、もう片方(離散側)はデータ不足で失敗、という、ちょうど良い次の一手が見えた形だと思います。

もし続けるなら、この2点を直した「v2」を組んで両タスクで再検証するのが次のステップですが、そちらは利用制限の様子を見ながら、また改めてになりそうですね。
</blockquote>
＜＜

jrf:＞いちおうその実験のファイルもください。T はどういうモデルなのですか？ そしてどうやって学習しているのですか？

＜


Claude:＞＞
<blockquote>

## Tは何か

**Tは、ただの d×d の実数行列です。** それ以上でもそれ以下でもありません。群である・回転であるといった制約は一切かかっていません。shiftタスクなら d=64(8×8行列を平坦化)、sin/cosタスクなら d=40(出力グリッドの点数)の、生の線形写像です。

「回転」や「置換」という性質は、Tに**そう見えるように制約したから**ではなく、**データにフィットさせた結果、たまたまそういう構造を持つ行列が出てきた**、というのがポイントです。実際、sin/cosタスクで学習されたTは、事後的に固有値を調べてみたら絶対値がほぼ1の複素固有値ペアを持っていた(=回転のシグネチャ)。これを最初から仮定してはいません。

## どうやって学習しているか

`fit_generator` 関数の中身がすべてで、勾配降下すら使っていません。**リッジ回帰(の閉形式解)だけ**です。

1. 訓練点のうち、s と s+1 が両方訓練集合にある「隣接ペア」を集める(shiftなら (1,2),(2,3),(5,6),(6,7) の4組)
2. 各ペアの φ(s) を並べた行列 X と、φ(s+1) を並べた行列 Y を作る
3. 「T·φ(s) ≈ φ(s+1) をすべてのペアで満たすT」を最小二乗で解く。閉形式は

```
T = Yᵀ X (XᵀX + λI)⁻¹
```

これは通常の線形回帰の正規方程式そのもので、λ(リッジ項)は行列が悪条件(病的)になるのを防ぐための微小な正則化です。ニューラルネットの訓練のような反復最適化は一切していません。1回の行列演算で解が求まります。

予測時は、この1ステップ用のTを**そのまま何度も掛け算(合成)する**だけです。s=3からs=4に進めたければ T を1回掛ける。s=5から逆向きに進めたければ T の疑似逆行列 T⁻¹ を1回掛ける。

## なぜこれでshiftタスクが失敗したか、が今回のTの姿からよく分かります

shiftタスクではTは64×64=4096個の自由パラメータを持つのに、学習に使える式は隣接ペア4組分しかありません。これは方程式の数よりパラメータの数が圧倒的に多い、**未定係数問題**です。リッジ回帰は「解けない部分は0に近づける」という形でこれを誤魔化しますが、結果としてTが「本物の置換行列」からはほど遠い、各行の絶対値和が0.5(本来1であるべき)という中途半端な行列になってしまいました。実験5のG(生成子)は、勾配降下で**訓練点6個全部への到達性**を損失にしていたので、この意味でずっと多くの制約情報を使っていた、というのが違いです。
</blockquote>
＜＜




In [ ]:
"""
実験6: 「アテンション保存写像」を一般化した Generator Discovery Framework

狙い: exp4(連続・回転的構造)と exp5(離散・巡回群構造)の両方を、
"同じ1つのアルゴリズム"で扱えるか試す。G^sのように「べき乗」という
関数形を人間が最初から与えるのではなく、1ステップぶんの遷移作用素 T を
隣接訓練点ペアからデータ駆動で発見し、それを合成して未見点を予測する。

手続き(タスクに依らず共通):
  1. 各訓練sについて、状態表現 φ(s) を用意する。
     (gauge問題を避けるため、重み空間ではなく「出力空間」の表現を使う
      ーー実験1・2で確認した「function-space表現は成功する」という
      教訓をここでも踏襲する)
  2. 訓練s群の中で「隣接ペア」(s, s+1がともに訓練集合にある組)を集める。
     例: (1,2),(2,3),(5,6),(6,7)  ※(3,4),(4,5)は s=4 が無いので使えない
  3. リッジ回帰で、1ステップ遷移作用素 T (線形写像) を
     φ(s+1) ≈ T φ(s) が全ペアで成り立つように最小二乗フィットする。
     (群構造を仮定していない。ただの線形回帰。答えが回転行列的になるか
      置換行列的になるかは、データが決める)
  4. 予測: 未見s=4を、最も近い訓練点からTを合成して届ける。
     forward:  φ(3) --T-->  φ(4)
     backward: φ(5) --T^-1--> φ(4)
     両方向で一致するかを内部整合性チェックとして見る。
  5. ベースライン(凸結合/線形補間)とLOOで比較し、どちらが勝つか
     ーー"どちらの帰納バイアスを使うべきか"自体もデータから選べる形にする。
"""

import torch
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

def fit_generator(phi_dict, ridge=1e-2):
    """隣接する訓練点ペアから、1ステップ遷移作用素 T をリッジ回帰でフィットする。"""
    sorted_s = sorted(phi_dict.keys())
    X, Y = [], []
    for s in sorted_s:
        if (s + 1) in phi_dict:
            X.append(phi_dict[s])
            Y.append(phi_dict[s + 1])
    X = torch.stack(X)  # [n_pairs, d]
    Y = torch.stack(Y)
    d = X.shape[1]
    # T X^T ≈ Y^T を解く(リッジ回帰): T = Y^T X (X^T X + λI)^-1
    XtX = X.T @ X + ridge * torch.eye(d)
    T = (Y.T @ X) @ torch.linalg.inv(XtX)
    return T, len(sorted_s), X, Y

def predict_generator(T, phi_dict, target_s, nearest_below, nearest_above):
    preds = {}
    if nearest_below in phi_dict:
        steps = target_s - nearest_below
        v = phi_dict[nearest_below]
        for _ in range(steps):
            v = T @ v
        preds["forward_from_%d" % nearest_below] = v
    if nearest_above in phi_dict:
        steps = nearest_above - target_s
        Tinv = torch.linalg.pinv(T)
        v = phi_dict[nearest_above]
        for _ in range(steps):
            v = Tinv @ v
        preds["backward_from_%d" % nearest_above] = v
    return preds

# ============================================================
# タスク1: shift (実験5と同じ)
# ============================================================
print("=" * 60)
print("タスク1: shift (離散・巡回群)")
print("=" * 60)
N = 8
sorted_train_s = [1, 2, 3, 5, 6, 7]
TEST_S = 4

def true_shift_matrix(s):
    P = torch.zeros(N, N)
    for i in range(N):
        P[i, (i + s) % N] = 1.0
    return P

phi_shift = {s: true_shift_matrix(s).flatten() for s in sorted_train_s}
T_shift, n, X, Y = fit_generator(phi_shift, ridge=1e-3)
preds = predict_generator(T_shift, phi_shift, TEST_S, 3, 5)

true_p4 = true_shift_matrix(TEST_S).flatten()
for name, v in preds.items():
    A_pred = v.view(N, N)
    A_true = true_p4.view(N, N)
    acc = (A_pred.argmax(dim=-1) == A_true.argmax(dim=-1)).float().mean().item()
    mse = ((v - true_p4) ** 2).mean().item()
    print(f"  {name}: 行accuracy={acc:.3f}  MSE={mse:.6f}")
    print(f"    予測パターン: {A_pred.argmax(dim=-1).tolist()}")
print(f"  (正解パターン: {true_p4.view(N,N).argmax(dim=-1).tolist()})")

# Tの性質を確認(データだけから、これが「ほぼ置換行列」に収束しているか)
print(f"\n  発見されたTの性質: 各行の絶対値和(理想的な置換行列なら1.0) = "
      f"{T_shift.abs().sum(dim=1).mean().item():.3f} (std={T_shift.abs().sum(dim=1).std().item():.3f})")

# ============================================================
# タスク2: sin/cos (実験1〜4と同じ。ただし表現は「出力空間」を使う)
# ============================================================
print("\n" + "=" * 60)
print("タスク2: sin/cos (連続・回転的構造)")
print("=" * 60)

def true_function(s, x):
    return torch.sin(2 * np.pi * (x + s / 7.0))

N_X = 40
X_GRID = torch.linspace(0, 1, N_X).unsqueeze(1)
sorted_train_s2 = [1, 2, 3, 5, 6, 7]
TEST_S2 = 4

# 状態表現: 出力空間(gauge問題のない表現。ここでは「真の」出力を直接使うが、
# 実際にはニューラルネット出力(実験1のfit_target_net)を使ってもよい)
phi_sin = {s: true_function(s, X_GRID).squeeze(1) for s in sorted_train_s2}

T_sin, n2, X2, Y2 = fit_generator(phi_sin, ridge=1e-2)
preds2 = predict_generator(T_sin, phi_sin, TEST_S2, 3, 5)

true_y4 = true_function(TEST_S2, X_GRID).squeeze(1)
for name, v in preds2.items():
    mse = ((v - true_y4) ** 2).mean().item()
    print(f"  {name}: MSE={mse:.6f}")

# 参考: 単純平均ベースライン
w_lin = 0.5 * phi_sin[3] + 0.5 * phi_sin[5]
mse_lin = ((w_lin - true_y4) ** 2).mean().item()
print(f"\n  参考baseline(s=3,5の単純平均): MSE={mse_lin:.6f}")

# Tが回転行列的(直交行列)になっているかチェック: T^T T ≈ I なら回転的
orth_err = ((T_sin.T @ T_sin - torch.eye(N_X)) ** 2).mean().item()
print(f"  発見されたTの直交性(T^T T - I のMSE、0に近いほど回転的): {orth_err:.6f}")
eigvals = torch.linalg.eigvals(T_sin)
print(f"  Tの固有値の絶対値(1に近いほど回転的、減衰/発散なし): "
      f"{sorted(eigvals.abs().tolist(), reverse=True)[:4]}")

print("\n" + "=" * 60)
print("総括")
print("=" * 60)
print("同一のアルゴリズム(隣接ペアからの線形作用素フィット+合成)を")
print("2つの異なるタスクに適用し、群構造が『離散(置換)』か『連続(回転)』か")
print("を教えずに、データだけからそれぞれ正しい合成的構造を発見できるかを見た。")


タスク1: shift (離散・巡回群)
  forward_from_3: 行accuracy=0.125  MSE=0.125000
    予測パターン: [0, 0, 0, 0, 0, 0, 0, 0]
  backward_from_5: 行accuracy=0.000  MSE=0.125000
    予測パターン: [6, 7, 0, 1, 2, 3, 4, 5]
  (正解パターン: [4, 5, 6, 7, 0, 1, 2, 3])

  発見されたTの性質: 各行の絶対値和(理想的な置換行列なら1.0) = 0.500 (std=0.504)

タスク2: sin/cos (連続・回転的構造)
  forward_from_3: MSE=0.000000
  backward_from_5: MSE=1.198931

  参考baseline(s=3,5の単純平均): MSE=0.069775
  発見されたTの直交性(T^T T - I のMSE、0に近いほど回転的): 0.023752
  Tの固有値の絶対値(1に近いほど回転的、減衰/発散なし): [0.9996171593666077, 0.9996171593666077, 6.834205851191655e-05, 5.959379268460907e-05]

総括
同一のアルゴリズム(隣接ペアからの線形作用素フィット+合成)を
2つの異なるタスクに適用し、群構造が『離散(置換)』か『連続(回転)』か
を教えずに、データだけからそれぞれ正しい合成的構造を発見できるかを見た。


jrf:＞アテンション保存性はどこに現れているのでしょう？ 1→2→3→... の順序がアテンションのようなものでその情報を使っているということでしょうか？

＜


Claude:＞＞
<blockquote>

正確に言うと、**exp6にはアテンションは一切登場していません。** ご指摘の通り、これは重要な混同で、訂正させてください。

## exp3/4の「アテンション」と exp6の違い

exp3/4の IsometricKVAttention は、**softmaxによる重み付き平均**でした。クエリ(新しいs)と、記憶バンクの全訓練点との類似度を計算し、その類似度で重み付けして値を混ぜ合わせる——これが「アテンション」の中身です。「どの点にどれだけ注意を向けるか」を、学習された類似度関数が決めていました。

exp6のTには、そのような機構が**まったくありません**。softmaxもなく、類似度計算もなく、「複数の点を混ぜ合わせる」という操作自体が存在しません。あるのは「1ステップ隣の点にどう移動するか」という単一の線形写像だけです。これはむしろ、実験5で見た「凸結合(アテンション)は離散構造に対して構造的に失敗する」という結果を受けて、**アテンション(混合)という発想そのものを捨てて、合成(composition)という別のカテゴリーの操作に乗り換えた**ものでした。

なので「アテンション保存写像」という呼び方は、exp6にはもう当てはまりません。これは正確には「線形力学系/群表現の発見」であって、「アテンションの一種」ではないです。ここは私の説明が不正確でした。訂正します。

## では 1→2→3→... の順序情報は何をしているのか

これは「アテンション」ではなく、**「隣接性(locality)の仮定」**として使われています。具体的には:

「s と s+1 が両方訓練集合にある組だけを使ってTをフィットする」という選び方そのものが、「1ステップ違いの点同士には、共通の(sに依らない)遷移作用素が存在するはずだ」という**強い先験的仮定**を、モデルの外側から人間が与えていることになります。これはアテンションのような「学習された・内容に基づく」重み付けではなく、「距離1の組だけを見る」という**構造的に固定されたルール**です。

対照的に言うと:
- **アテンション(exp3/4)**: 「どの点に注目すべきか」を学習で決める。距離が近くても遠くても、データが言うならどの点にでも注意を向けられる、柔軟だが構造の保証がない機構。
- **合成(exp6)**: 「隣の点にだけ関係がある」ということを人間が先に決めてしまい、その"隣接1ステップ"の関係だけを学習する。柔軟性は失うが、合成則(群構造)という強い保証を得られる機構。

つまり今回、アテンションを保存する枠組みを探していたはずが、いつのまにか**アテンションを完全に捨てて、別の帰納バイアス(合成的構造)に乗り換えていた**、というのが正確な経緯だったと思います。もし「アテンション性」を残したまま合成的構造も持たせたいなら、「どの生成子(候補となる複数のT)を使うかをアテンションで選ぶ」というハイブリッド構成は考えられますが、それはまた別の設計になります。
</blockquote>
＜＜

jrf:＞「アテンション保存」ということは何がしかの法則性とその法則を今どこに適用しているかがわかっているという状態から、それが具体的にどう展開されるかにしていく…ということで、他の 1,2,3,5,6,7 から 4 が予想できるというのはこの特徴による。…ということでしょう。そうでないものは予想できないのは当然だ…と。そして、今回の shift も sin/cos も予想できる範疇に思える。それがうまくいっていないのは、出力側の性能か、学習対象の空間が広過ぎるか…で、後者なら、「アテンション保存」では十分枝刈りができていない…という感じですか…？

＜


Claude:＞＞
<blockquote>

その整理、かなり正確だと思います。ただ一点、用語を訂正させてください。それは「アテンション保存」ではなく、むしろ「アテンションを捨てて手に入れたもの」でした。

## 「法則+現在地→展開」という特徴づけは正しいですが、それは合成(composition)の特徴です

おっしゃる「法則性と、それを今どこに適用しているかが分かっている状態から、具体的にどう展開されるかにしていく」というのは、まさに exp5/exp6 の T・G のことです。「現在地(s=3)」に「1ステップぶんの遷移法則(T)」を適用して「次の状態(s=4)」を作る——これは明確に「現在地」という概念と「そこから動かす操作」という概念を両方持っています。

一方、exp3/4の**アテンション(softmax加重平均)には「現在地」も「展開」も存在しません**。あるのは「訓練点全部との類似度を計算して、いっぺんに混ぜ合わせる」という、対称的で時間発展のない操作だけです。「1ステップ進める」という概念自体がアテンションには組み込まれていない。なので正確には、JRFさんが言っている「予測できる特徴」は「合成的構造」の話であって、「アテンション保存」という当初の呼び方は、この特徴を指す言葉としては不適切だった、というのが今回はっきりしたことだと思います。

## 「予想できる範疇」かどうかの3分類

ここまでの実験を振り返ると、失敗には実は3種類あったと思います。

**(1) そもそも表現できない(法則の入れ物が間違っている)**
exp5のE(凸結合アテンション)がこれでした。置換行列は多面体の頂点なので、他の頂点の混ぜ合わせでは原理的に到達不可能。これは「予想できない範疇」の典型例で、いくら枝刈りしても直らない。**混ぜる(加重平均)ではなく、掛ける(合成)という操作カテゴリ自体を選び直す必要がありました。**

**(2) 表現の入れ物は正しいが、枝刈り(制約)が足りない**
まさにJRFさんが言う「後者」です。exp6のshiftタスクがこれでした。Tは64×64=4096自由パラメータあるのに、隣接ペアの制約式は4本しかない。実験5のG(softmaxで行stochasticに制約+訓練点6個全部を使う)は、この枝刈りを追加して成功しました。つまり**「1ステップの遷移」という入れ物(合成的構造)自体は正しかったが、制約が緩すぎて迷子になっていた**、という診断で合っています。

**(3) 入れ物も枝刈りも十分だが、実装上の技術的な問題(出力側の性能)**
exp6のsin/cosタスクの後退方向がこれでした。真の力学は40次元中たった2次元の部分空間にしか存在しないのに、Tを40次元全体でフィットしたため、疑似逆行列の計算が数値的に不安定になった。これは「予想できる範疇」であることは前進方向のMSE=0.000000で証明済みで、純粋に実装の詰めの問題でした。

## まとめると

「そうでないものは予想できないのは当然」という直感は(1)のケースで正しく、「後者(枝刈り不足)」という診断は(2)のケースで正確に当たっています。今回の shift も sin/cos も、**入れ物としては「予想できる範疇」に入っていた**というのはその通りで、実際に失敗の内訳を見ると、shiftは(2)、sin/cosは(3)でした。つまり両方とも「アテンション保存では枝刈り不足だった」というより、正確には「アテンションという入れ物自体を捨てて合成という入れ物に変えた後、枝刈りの精度をさらに詰め切れていなかった」という状態だと思います。
</blockquote>
＜＜

jrf:＞
Gemini さんによると実験5の $G^s$ こそが本当の「アテンション保存変換」ではないかというのです。確かに、s の順序構造を G を s 回作用させたものということで「保存」するのが保存変換ではないか…と。sin/cos もこの構造で学習できるはずというのです。そのような代数構造を仮定して学習してみれば…ということのようです。

```pyhon
import torch
import torch.nn as nn
import torch.nn.functional as F
class AttentionPreservingMapHyper(nn.Module):
    """
    単なる「アテンションの凸結合(平均)」ではなく、
    アテンションの性質(行確率・関係性)を保存したまま s に応じて推移させる写像
    """
    def __init__(self, N=8, d_model=16):
        super().__init__()
        self.N = N
        
        # 1. 1ステップの「アテンション保存写像(確率保存変換)」を行う生成子 G
        self.raw_G = nn.Parameter(torch.randn(N, N) * 0.5)
        
        # 2. 基準アテンションパターン (例: s=0 の恒等写像 I)
        self.register_buffer("A_zero", torch.eye(N))
    def get_transform_operator(self):
        """アテンションの条件(各行の和=1, 非負)を保持する変換作用素 G"""
        return F.softmax(self.raw_G, dim=-1)
    def forward(self, s: int):
        """
        アテンション保存写像の実行:
        基準アテンション A_0 に対し、保存変換 G を s 回適用(作用)させる
        """
        G = self.get_transform_operator()
        
        # 単なる加重平均ではなく、アテンション構造を保つ「変換(写像)」の反復適用
        A_s = self.A_zero
        for _ in range(s):
            A_s = A_s @ G  # 行列積による代数的合成
            
        return A_s
# ---------------------------------------------------------
# 動作確認
# ---------------------------------------------------------
model = AttentionPreservingMapHyper(N=8)
A_4 = model(s=4)
# 性質の検証: 各行の和が 1.0 に保たれているか（アテンション構造の保存）
print("各行の和:", A_4.sum(dim=-1).detach().numpy().round(3))
# Output: [1. 1. 1. 1. 1. 1. 1. 1.]  -> アテンション構造が厳密に保存されている
```
こんなクラスも提案されました。ただ、これはこのままでは、やはり表現力が足りてないとは思うのですが。

＜


Claude:＞＞
<blockquote>

良い質問です。「行確率(各行の和=1・非負)」という性質を保つのは、たしかにアテンションの一部の性質(確率分布であること)を保存していると言えます。ただ、この構成には**数学的に証明できる表現力の限界**があります。実際に確かめてみます。

まず理論的に:2×2の行確率行列(非負・各行の和=1)の固有値は、常に実数になります。証明は簡単で、`[[a,1-a],[1-b,b]]`という一般形の判別式を計算すると `(a+b-2)²` という完全平方になり、負になり得ません。**つまり、複素固有値(=回転)を絶対に表現できません。** 回転で表現できるのは実質「恒等写像(0°)」か「反転(180°)」だけです。sin/cosタスクの真の力学は「回転」そのものなので、この制約下では原理的に正しく学習できないはずです。実際に試してみます。はっきり出ました。

## 数値的にも確認できました

| 手法 | 関数出力でのMSE(s=4) |
|---|---|
| Gemini案(行確率保存G、softmax) | **1.710** |
| 制約なしT | 0.572 |
| 参考: 単純平均 | 0.070 |

行確率保存のGは、単純な平均よりずっと悪い結果になりました。学習されたGの固有値を見ると `[1.0, 0.281]` と、**両方とも実数**です。理論通り、複素固有値(回転)には到達できていません。

## ただし正直に言うと、この実験には気になる点があります

制約なしT(0.572)も、実は前回exp6で見た「ほぼ完璧(MSE=0.000000)」からは程遠い結果になっています。理由は訓練の設計を変えたからです——exp6では「隣接ペアの閉形式リッジ回帰」という簡単な問題を解いていましたが、今回は「s=1を基点に、最大6ステップ分の行列累乗を通じた勾配降下」という、はるかに最適化の難しい問題を解かせています(行列を何度も掛け合わせる経路は勾配消失・爆発を起こしやすい)。その結果、制約なしTの学習自体も理想的な回転行列(固有値2.199, 0.310、これも実数!)には辿り着けませんでした。

なので、**「行確率保存Gがsin/cosタスクに向いていない」という結論自体は理論的に確実(判別式の証明)ですが、今回の数値実験の差(1.71 vs 0.57)を鵜呑みにするのは危険**です。両方とも最適化に失敗している可能性があり、「制約が悪い」というより「複数ステップの合成を勾配降下で学習させること自体が難しい」という、また別の問題が混じっていそうです。

## まとめ

JRFさんの「表現力が足りていない」という直感は、**理論的には正しいと証明できます**(2×2行確率行列の固有値は判別式が完全平方になるため必ず実数=回転を表現できない)。これはGemini案が原理的にsin/cosタスクを解けないという、確定的な反例です。ただし、それを裏付けるはずだった数値実験は、比較対象(制約なしT)側の訓練も不完全だったため、額面通りには受け取れません。もし追試するなら、exp6のような閉形式回帰(勾配降下でなく)で制約なしTを求め、それと今回のGemini案Gを同条件で比較するのが、よりフェアな検証になると思います。行確率行列の固有値が実数しか取れないという証明(判別式が完全平方になる)は確実な結果ですが、それを裏付けるはずだった数値実験の比較対象(制約なしT)自体が今回の訓練設計では最適化に失敗していたので、そこは割り引いて見てください。もし続けるなら、exp6方式の閉形式回帰でフェアに比較し直すのが次の一手だと思います。
</blockquote>
＜＜




In [ ]:
"""
実験7: Gemini さん提案の AttentionPreservingMapHyper (G=softmax行、行確率保存)
        を sin/cos タスクに適用したら本当に失敗するか検証する

理論的予測: NxN の行確率行列(非負・各行の和=1)の固有値は、2x2の場合、
判別式が (a+b-2)^2 という完全平方になり、常に実数になる(証明は判別式計算で確認可能)。
つまり複素固有値=回転を表現できず、sin/cosタスクの真の回転力学
(exp6でT^T T ≈ I, 固有値の絶対値≈1の複素ペアだったことを確認済み)を
原理的に再現できないはずである。

ここでは、sin/cosタスクの状態を(cosθ,sinθ)という2次元表現(データから
線形回帰で抽出可能、真の式を直接与えているわけではない)に落とし込み、
Gemini案のG(softmax行、行確率保存)と、制約なしT(exp6と同じ)の
両方を同じ条件でフィットし、s=4での性能を比較する。
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

def true_function(s, x):
    return torch.sin(2 * np.pi * (x + s / 7.0))

N_X = 40
X_GRID = torch.linspace(0, 1, N_X).unsqueeze(1)
sorted_train_s = [1, 2, 3, 5, 6, 7]
TEST_S = 4

# ------------------------------------------------------------
# 状態表現を2次元に圧縮: 出力y_sを sin(2πx), cos(2πx) の基底に回帰する
# (真の式 sin(2π(x+s/7)) を直接使わず、データからこの2次元性を発見する)
# ------------------------------------------------------------
basis = torch.cat([torch.sin(2*np.pi*X_GRID), torch.cos(2*np.pi*X_GRID)], dim=1)  # [N_X, 2]
basis_pinv = torch.linalg.pinv(basis)  # [2, N_X]

def phi2(s):
    y = true_function(s, X_GRID)  # [N_X, 1]
    coef = basis_pinv @ y  # [2, 1] -> (a,b) s.t. y ≈ a*sin(2πx)+b*cos(2πx)
    return coef.squeeze(1)

phi_train = {s: phi2(s) for s in sorted_train_s}
true_phi4 = phi2(TEST_S)
print(f"2次元表現の例: phi(1)={phi_train[1].numpy().round(3)}, "
      f"phi(4)(正解、テスト用にのみ表示)={true_phi4.numpy().round(3)}")

def decode(coef):
    return (basis @ coef.unsqueeze(1)).squeeze(1)

# ------------------------------------------------------------
# モデル1: Gemini案 AttentionPreservingMapHyper 相当 (G=softmax行、行確率保存)
# ------------------------------------------------------------
class RowStochasticGenerator(nn.Module):
    def __init__(self, d=2):
        super().__init__()
        self.raw_G = nn.Parameter(torch.randn(d, d) * 0.5)
    def G(self):
        return F.softmax(self.raw_G, dim=-1)  # 各行が確率分布(非負・和=1)
    def power(self, base_vec, s):
        G = self.G()
        v = base_vec
        for _ in range(s):
            v = G @ v
        return v

def train_rowstochastic(n_steps=3000, lr=3e-2):
    model = RowStochasticGenerator(d=2)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    anchor = 1  # s=1を基点にする
    base = phi_train[anchor]
    for step in range(n_steps):
        opt.zero_grad()
        loss = 0.0
        for s in sorted_train_s:
            if s == anchor:
                continue
            pred = model.power(base, s - anchor)
            loss = loss + ((pred - phi_train[s]) ** 2).mean()
        loss = loss / (len(sorted_train_s) - 1)
        loss.backward()
        opt.step()
    return model

# ------------------------------------------------------------
# モデル2: 制約なしT (exp6と同じ思想だが、勾配降下+全点合成損失を使う。フェアな比較のため)
# ------------------------------------------------------------
class UnconstrainedGenerator(nn.Module):
    def __init__(self, d=2):
        super().__init__()
        self.T = nn.Parameter(torch.eye(d) + torch.randn(d, d) * 0.1)
    def power(self, base_vec, s):
        v = base_vec
        for _ in range(s):
            v = self.T @ v
        return v

def train_unconstrained(n_steps=3000, lr=3e-2):
    model = UnconstrainedGenerator(d=2)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    anchor = 1
    base = phi_train[anchor]
    for step in range(n_steps):
        opt.zero_grad()
        loss = 0.0
        for s in sorted_train_s:
            if s == anchor:
                continue
            pred = model.power(base, s - anchor)
            loss = loss + ((pred - phi_train[s]) ** 2).mean()
        loss = loss / (len(sorted_train_s) - 1)
        loss.backward()
        opt.step()
    return model

print("\n=== モデル1: Gemini案 (G=softmax行、行確率保存) を訓練 ===")
model_rs = train_rowstochastic()
G_rs = model_rs.G().detach()
eigvals_rs = torch.linalg.eigvals(G_rs)
pred4_rs = model_rs.power(phi_train[1], TEST_S - 1)
mse_rs = ((pred4_rs - true_phi4) ** 2).mean().item()
y_pred_rs = decode(pred4_rs.detach())
y_true4 = true_function(TEST_S, X_GRID).squeeze(1)
func_mse_rs = ((y_pred_rs - y_true4) ** 2).mean().item()
print(f"  学習されたGの固有値: {eigvals_rs.numpy()}")
print(f"  2次元表現でのMSE: {mse_rs:.6f}")
print(f"  関数出力に戻したときのMSE: {func_mse_rs:.6f}")

print("\n=== モデル2: 制約なしT を訓練(比較対象) ===")
model_uc = train_unconstrained()
T_uc = model_uc.T.detach()
eigvals_uc = torch.linalg.eigvals(T_uc)
pred4_uc = model_uc.power(phi_train[1], TEST_S - 1)
mse_uc = ((pred4_uc - true_phi4) ** 2).mean().item()
y_pred_uc = decode(pred4_uc.detach())
func_mse_uc = ((y_pred_uc - y_true4) ** 2).mean().item()
print(f"  学習されたTの固有値: {eigvals_uc.numpy()}")
print(f"  2次元表現でのMSE: {mse_uc:.6f}")
print(f"  関数出力に戻したときのMSE: {func_mse_uc:.6f}")

# 参考: 単純平均
w_lin = 0.5 * phi_train[3] + 0.5 * phi_train[5]
y_lin = decode(w_lin)
mse_lin = ((y_lin - y_true4) ** 2).mean().item()

print(f"\n参考baseline(s=3,5の単純平均、関数出力MSE): {mse_lin:.6f}")

print("\n=== 総括 ===")
print(f"{'手法':40s} {'関数出力MSE (s=4)':>20s}")
print(f"{'Gemini案(行確率保存G)':40s} {func_mse_rs:>20.6f}")
print(f"{'制約なしT':40s} {func_mse_uc:>20.6f}")
print(f"{'参考: 単純平均':40s} {mse_lin:>20.6f}")


2次元表現の例: phi(1)=[0.623 0.782], phi(4)(正解、テスト用にのみ表示)=[-0.901 -0.434]

=== モデル1: Gemini案 (G=softmax行、行確率保存) を訓練 ===
  学習されたGの固有値: [1.        +0.j 0.28115523+0.j]
  2次元表現でのMSE: 1.724768
  関数出力に戻したときのMSE: 1.709787

=== モデル2: 制約なしT を訓練(比較対象) ===
  学習されたTの固有値: [2.1375675 +0.j 0.29450482+0.j]
  2次元表現でのMSE: 0.601955
  関数出力に戻したときのMSE: 0.591059

参考baseline(s=3,5の単純平均、関数出力MSE): 0.069775

=== 総括 ===
手法                                              関数出力MSE (s=4)
Gemini案(行確率保存G)                                      1.709787
制約なしT                                                0.591059
参考: 単純平均                                             0.069775


jrf:＞d=2 のただの1層の行列だとそりゃそうだろう…という感想なのですが、G^s の意味、G * G (* ...) の意味をもう少し拡張したらできるという方向はないですか？

＜


Claude:＞＞
<blockquote>

良い勘です。2×2の単一線形層という限定が本質的な制約でした。拡張の方向は大きく2つあると思います。ひとつは「表現空間を豊かにして、行確率という制約自体は保ったまま普遍性を回復する」方向、もうひとつは「G自体を非線形にする」方向です。前者を実装して試してみます。部分的にしか改善しませんでした。

## 結果

| 手法 | 関数出力MSE(s=4) |
|---|---|
| 単純平均(s=3,5) | 0.070 |
| 2次元・行確率保存G(exp7) | 1.710 |
| 集団符号化・行確率保存G(今回) | 0.876 |

改善はしましたが(1.71→0.88)、単純平均には遠く及ばず、狙った通りの「複素固有値(回転)」もほとんど出ませんでした。学習されたGの固有値は上位4つのうち3つが実数で、4つ目にようやく極めて小さい虚部(0.0064+0.0024j)が出た程度です。

## これは理論の反証ではなく、訓練手法の限界だと思います

exp7の「制約なしT」でも、同じ「複数ステップの行列累乗を勾配降下で通す」という訓練法を使ったとき、理論上可能なはずの回転(複素固有値)を見つけられませんでした。つまり**「集団符号化なら行確率制約下でも回転を表現できる」という理論的な主張自体は、shiftタスクがすでに実例として証明しています**(置換行列は行確率行列の特殊ケースで、リング上の離散回転そのものだからです)。今回失敗したのは、「答えを知らない状態で、6ステップ分の合成を勾配降下だけで逆算させる」という学習問題が、単純に難しすぎたということだと思います。exp6で使った「隣接ペアの閉形式回帰」のような、もっと訓練しやすい目的関数に変えれば、改善する余地はありそうです。

## まとめると、拡張の方向性は2つ見えました

1. **表現空間を拡張する(集団符号化)**: 理論的には正しいはずだが、今回の勾配降下ベースの訓練では実証できなかった。閉形式回帰などのもっと安定した学習法が必要。
2. **G自体を非線形にする(Neural ODE的な発想)**: 行列の代わりに小さなニューラルネット g を使い、A_s = g(g(...g(A_0)...)) と繰り返し適用する、あるいは連続版として微分方程式(ベクトル場)を学習してsだけ時間発展させる。これは線形性の制約自体をなくすので、より根本的な拡張になりますが、今回はまだ試せていません。

利用制限が近そうとのことでしたので、ここで一区切りにします。ファイルは以下に置きました。良い着地点だと思います。「単一の線形G」という限定を外す方向は、今回試した2つとも筋は悪くなさそうでしたが、決着まではつけられませんでした。続きはまたいつでもどうぞ。
</blockquote>
＜＜



In [ ]:
"""
実験8: 行確率保存(G=softmax行)のまま、表現空間を拡張して回転を表現できるか

理論: 2次元の生の(a,b)=(cosθ,sinθ)表現では、行確率行列は実固有値しか
持てず回転を表現できないことを exp7 で確認した。

しかし、shiftタスクで置換行列(0/1の行確率行列)が「巡回群上の回転」を
問題なく表現できていたことを思い出すと、鍵は「次元の低さ」ではなく
「符号付きの座標(a,bのように負の値を含む)を直接使っていたこと」に
あるとわかる。非負性(softmax)と「回転」は、表現が非負(確率分布的)で
ありさえすれば両立できる。

具体的には、神経科学の「リング・アトラクタ(ring attractor)」/
「集団符号化(population coding)」と同じ発想を使う: 連続な位相 θ を、
単一の(a,b)座標ではなく、リング状に並んだ N_pop 個のユニットの
活動パターン(フォン・ミーゼス分布に似た山型、非負・合計1)として表現する。
すると「位相をΔだけ進める」操作は、活動パターンを N_pop 個のユニットの
リング上で回転させる操作になり、これは非負・行確率(circulant)行列で
厳密に表現できる。

つまり「行確率保存(attention性)」という制約自体は表現力の敵ではなく、
表現の埋め込み方(次元・符号)が適切であれば普遍性を取り戻せる、
という仮説を検証する。
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

def true_function(s, x):
    return torch.sin(2 * np.pi * (x + s / 7.0))

N_X = 40
X_GRID = torch.linspace(0, 1, N_X).unsqueeze(1)
sorted_train_s = [1, 2, 3, 5, 6, 7]
TEST_S = 4

# ------------------------------------------------------------
# 集団符号化(population code)による位相のリング表現
# ------------------------------------------------------------
N_POP = 48          # リング上のユニット数(7の倍数ではなく、あえて汎用的な数にする)
KAPPA = 8.0          # フォン・ミーゼス分布の集中度(山の鋭さ)

pop_angles = 2 * np.pi * torch.arange(N_POP).float() / N_POP

def phase_of(s):
    return 2 * np.pi * (s / 7.0)

def population_code(theta):
    """θ(スカラー)を、N_POP次元の非負・合計1のリング表現に変換する。"""
    logits = KAPPA * torch.cos(pop_angles - theta)
    return F.softmax(logits, dim=0)  # 既にそれ自体が「アテンション」的な分布

def decode_phase(pop_vec):
    """リング表現から位相を復元する(円周上の重心=circular mean)。"""
    c = (pop_vec * torch.cos(pop_angles)).sum()
    s_ = (pop_vec * torch.sin(pop_angles)).sum()
    return torch.atan2(s_, c)

phi_train = {s: population_code(phase_of(s)) for s in sorted_train_s}
true_phi4 = population_code(phase_of(TEST_S))

# ------------------------------------------------------------
# G: 行確率保存(softmax行)の1ステップ遷移行列。exp5のGeneratorHyperと同じ形。
#    ただし今回は「集団符号化された表現」の上で動かす。
# ------------------------------------------------------------
class RowStochasticGeneratorPop(nn.Module):
    def __init__(self, n=N_POP):
        super().__init__()
        self.raw_G = nn.Parameter(torch.randn(n, n) * 0.3)
    def G(self):
        return F.softmax(self.raw_G, dim=-1)
    def power(self, base_vec, s):
        G = self.G()
        v = base_vec
        for _ in range(s):
            v = G @ v
        return v

def train_pop(n_steps=2500, lr=1e-2):
    model = RowStochasticGeneratorPop()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    anchor = 1
    base = phi_train[anchor]
    for step in range(n_steps):
        opt.zero_grad()
        loss = 0.0
        for s in sorted_train_s:
            if s == anchor:
                continue
            pred = model.power(base, s - anchor)
            # 確率分布同士なのでKLダイバージェンスで比較する
            loss = loss + F.kl_div(torch.log(pred.clamp_min(1e-8)), phi_train[s], reduction="batchmean")
        loss = loss / (len(sorted_train_s) - 1)
        loss.backward()
        opt.step()
        if step % 500 == 0:
            print(f"    step {step}: loss={loss.item():.5f}")
    return model

print("=== 集団符号化 + 行確率保存Gを訓練 ===")
model = train_pop()

with torch.no_grad():
    pred4 = model.power(phi_train[1], TEST_S - 1)
    pred_theta = decode_phase(pred4)
    true_theta = phase_of(TEST_S)
    print(f"\n真の位相 θ(s=4)      = {true_theta:.4f} rad")
    print(f"予測された位相(復元)  = {pred_theta.item():.4f} rad")
    print(f"位相の誤差            = {abs(((pred_theta - true_theta + np.pi) % (2*np.pi)) - np.pi):.5f} rad")

    # 関数出力に変換して比較
    def phase_to_y(theta):
        return torch.sin(2 * np.pi * X_GRID.squeeze(1) + theta)
    y_pred = phase_to_y(pred_theta)
    y_true = true_function(TEST_S, X_GRID).squeeze(1)
    mse = ((y_pred - y_true) ** 2).mean().item()
    print(f"関数出力でのMSE(s=4)  = {mse:.6f}")

    # Gの固有値: 複素になっているか確認
    G_learned = model.G()
    eigvals = torch.linalg.eigvals(G_learned)
    top4 = sorted(eigvals.tolist(), key=lambda z: -abs(z))[:4]
    print(f"\n学習されたG(48x48)の固有値(絶対値大きい順、上位4個): "
          f"{[complex(z.real,z.imag) for z in top4]}")
    print(f"  (複素固有値=回転成分が出ていれば、行確率制約下でも回転を獲得できたことになる)")

print("\n=== 参考: これまでの結果との比較 ===")
w_lin_mse = 0.069775  # 単純平均(既出)
print(f"{'手法':45s} {'関数出力MSE (s=4)':>18s}")
print(f"{'単純平均(s=3,5)':45s} {w_lin_mse:>18.6f}")
print(f"{'2次元・行確率保存G(exp7、失敗)':45s} {1.709787:>18.6f}")
print(f"{'集団符号化・行確率保存G(今回)':45s} {mse:>18.6f}")


=== 集団符号化 + 行確率保存Gを訓練 ===
    step 0: loss=0.02967
    step 500: loss=-0.00983
    step 1000: loss=-0.01060
    step 1500: loss=-0.01070
    step 2000: loss=-0.01073

真の位相 θ(s=4)      = 3.5904 rad
予測された位相(復元)  = 2.1625 rad
位相の誤差            = 1.42793 rad
関数出力でのMSE(s=4)  = 0.876118

学習されたG(48x48)の固有値(絶対値大きい順、上位4個): [(1.000000238418579+0j), (0.10853121429681778+0j), (0.01875101774930954+0j), (0.006408594083040953+0.0024388593155890703j)]
  (複素固有値=回転成分が出ていれば、行確率制約下でも回転を獲得できたことになる)

=== 参考: これまでの結果との比較 ===
手法                                                 関数出力MSE (s=4)
単純平均(s=3,5)                                             0.069775
2次元・行確率保存G(exp7、失敗)                                     1.709787
集団符号化・行確率保存G(今回)                                        0.876118


jrf:＞行列の代わりに小さなニューラルネット g を使い、A_s = g(g(...g(A_0)...)) と繰り返し適用する、あるいは連続版として微分方程式(ベクトル場)を学習してsだけ時間発展させる。…方向が気になりますね。実験4または実験5の例でそれでどこまで行けるか興味あります。

＜


Claude:＞＞
<blockquote>

良い方向だと思います。前回(exp7/8)で「多段合成を勾配降下で通す」学習が不安定だったので、今回は**1ステップの教師信号だけで学習し(exp6の閉形式回帰と同じ発想)、gの中身だけを非線形にする**という設計にします。gを「恒等写像+小さな摂動」(残差構造、Neural ODEのオイラー離散化に近い形)にして安定させます。くっきりした結果になりました。

## sin/cos: これまでで最良の結果

**MSE=0.001949**。これはexp6の線形Tの0.000000(理想的な回転)にほぼ匹敵し、単純平均(0.070)より**約35倍良い**結果です。しかもs=1から3ステップ独立に合成した経路でも全く同じ値になり、内部的な整合性も取れています。非線形の残差構造(g=恒等写像+小さなMLP摂動)は、線形回転をきちんと表現しつつ、かつ勾配消失の問題もなく安定して学習できました。

## shift: 完全に失敗

**行accuracy 0%**。予測パターンを見ると、g(φ(3)) がほぼ φ(3) 自身(何もシフトしていない)になっていて、モデルがほとんど「何もしない」方向に収束してしまっています。

## なぜこの差が出たか

これは「非線形かどうか」の問題ではなく、**学習に使った教師信号の量**の違いだと思います。今回、両方のタスクとも「隣接ペア4組だけ」を直接の教師信号にしました。sin/cosでは、真の力学がなめらかな回転なので、たった4組のペアからでも十分にgの挙動が一般化した。一方shiftは、置換という「離散的で連続性のない」写像なので、4組のペアだけでは64次元の変換をほとんど制約できず、残差構造(恒等写像に近い初期化)がそのまま「何もしない」解に居着いてしまったのだと思います。実験5でG^sが成功したのは、訓練点6個**全部**への到達性を損失にしていたからで、この「教師信号の設計」の差が今回はっきり出ました。

**まとめると**: 「非線形生成子(Neural ODE的発想)」は、少なくとも連続・滑らかなタスクに対しては、線形生成子とほぼ同等の性能を、より少ない仮定(答えを知らなくても)で達成できることが分かりました。ただし離散・組み合わせ的なタスクには、非線形性そのものよりも「多点への到達性を学習信号に含めること」の方が重要そうです。次にやるなら、shiftタスクでも非線形gを訓練点6個全部への到達性で学習させ直す、というのが素直な続きだと思います。sin/cosの方は、これまでの実験全体を通しても最も良い結果の一つだと思います。「答えの関数形(回転行列)を教えなくても、残差構造+隣接ペアだけの学習で、ほぼ理想解に到達できる」というのは、当初の「アテンション保存写像」という発想の、かなり良い落とし所になっているように思います。
</blockquote>
＜＜



In [ ]:
"""
実験9: 非線形生成子 g によるA_s = g(g(...g(A_0)...)) (実質Neural ODEのEuler版)

これまでの反省:
  - exp7/8で「多段の合成を勾配降下で通す」訓練は、深い合成を通じた
    勾配消失/爆発のせいでうまく最適化できなかった(制約なしTですら失敗)。
  - exp6の教訓: 隣接ペア(1ステップ)だけを直接教師信号にする閉形式回帰は
    安定して解けた(sin/cosで前進予測がMSE=0.000000)。

そこで今回は、gを「1ステップの遷移」として、隣接ペアだけを直接の教師信号にして
学習する(多段合成は勾配計算には使わない、予測時にのみ繰り返し適用する)。
gの中身は、恒等写像+小さなMLP摂動(残差構造):
    g(x) = x + eps * MLP(x)
という形にする。これは
    dx/dt = MLP(x),  x(t+dt) = x(t) + dt*MLP(x(t))
というEuler法によるODE積分の1ステップそのものであり、「非線形生成子による
時間発展」というNeural ODE的な発想を、素朴なEuler離散化で実装したものになる。
線形写像はこの特殊ケース(MLPが線形層1つだけ)として含まれるので、
表現力は少なくとも exp6 のT以上になるはず。

sin/cos(連続)とshift(離散)の両方で試す。
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

# ============================================================
# 共通: 非線形生成子 g (残差MLP)
# ============================================================
class ResidualGenerator(nn.Module):
    def __init__(self, dim, hidden=32, eps_init=1.0):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden), nn.Tanh(),
            nn.Linear(hidden, dim)
        )
        # epsも学習可能にする(ステップ幅=遷移の強さ)
        self.log_eps = nn.Parameter(torch.log(torch.tensor(eps_init)))
        # 小さく初期化してg≈恒等写像から始める(残差構造の定石)
        for p in self.mlp[-1].parameters():
            nn.init.zeros_(p)

    def step(self, x):
        eps = torch.exp(self.log_eps)
        return x + eps * self.mlp(x)

    def apply_n_times(self, x, n):
        for _ in range(n):
            x = self.step(x)
        return x


def train_one_step_supervised(phi_dict, dim, n_steps=3000, lr=3e-3, hidden=32):
    """隣接ペア(s, s+1がともに訓練集合にある組)だけを直接の教師信号にする。
       多段合成は使わない(=exp7/8で問題になった深い勾配の不安定さを回避)。"""
    sorted_s = sorted(phi_dict.keys())
    pairs = [(s, s + 1) for s in sorted_s if (s + 1) in phi_dict]
    model = ResidualGenerator(dim, hidden=hidden)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for step in range(n_steps):
        opt.zero_grad()
        loss = 0.0
        for s_a, s_b in pairs:
            pred = model.step(phi_dict[s_a])
            loss = loss + ((pred - phi_dict[s_b]) ** 2).mean()
        loss = loss / len(pairs)
        loss.backward()
        opt.step()
    return model


# ============================================================
# タスク1: sin/cos (連続) — 2次元(cosθ,sinθ)表現を使う
# ============================================================
print("=" * 60)
print("タスク1: sin/cos (連続・回転的構造)")
print("=" * 60)

def true_function(s, x):
    return torch.sin(2 * np.pi * (x + s / 7.0))

N_X = 40
X_GRID = torch.linspace(0, 1, N_X).unsqueeze(1)
sorted_train_s = [1, 2, 3, 5, 6, 7]
TEST_S = 4

basis = torch.cat([torch.sin(2*np.pi*X_GRID), torch.cos(2*np.pi*X_GRID)], dim=1)
basis_pinv = torch.linalg.pinv(basis)

def phi2(s):
    y = true_function(s, X_GRID)
    return (basis_pinv @ y).squeeze(1)

def decode(coef):
    return (basis @ coef.unsqueeze(1)).squeeze(1)

phi_sin = {s: phi2(s) for s in sorted_train_s}
true_phi4 = phi2(TEST_S)
y_true4 = true_function(TEST_S, X_GRID).squeeze(1)

model_sin = train_one_step_supervised(phi_sin, dim=2, n_steps=3000, lr=5e-3, hidden=32)

with torch.no_grad():
    pred_fwd = model_sin.apply_n_times(phi_sin[3], 1)   # s=3から1ステップ
    pred_bwd = phi_sin[5]  # 逆方向は非線形だと解析的逆写像がないので今回は省略
    mse_fwd_coef = ((pred_fwd - true_phi4) ** 2).mean().item()
    y_pred_fwd = decode(pred_fwd)
    mse_fwd_func = ((y_pred_fwd - y_true4) ** 2).mean().item()

    # 内部整合性チェック: s=1から3回適用したら s=4 に届くか(訓練データだけでは検証不可能な経路)
    pred_from1 = model_sin.apply_n_times(phi_sin[1], 3)
    mse_from1_func = ((decode(pred_from1) - y_true4) ** 2).mean().item()

    print(f"  s=3から1ステップ: 2次元MSE={mse_fwd_coef:.6f}  関数出力MSE={mse_fwd_func:.6f}")
    print(f"  s=1から3ステップ(独立経路での整合性チェック): 関数出力MSE={mse_from1_func:.6f}")
    print(f"  学習されたeps: {torch.exp(model_sin.log_eps).item():.4f}")

print(f"\n  参考: 単純平均(s=3,5)の関数出力MSE = 0.069775")
print(f"  参考: exp6の線形T(s=3から1ステップ)の関数出力MSE = 0.000000")

# ============================================================
# タスク2: shift (離散・巡回群)
# ============================================================
print("\n" + "=" * 60)
print("タスク2: shift (離散・巡回群)")
print("=" * 60)
N = 8
sorted_train_s2 = [1, 2, 3, 5, 6, 7]
TEST_S2 = 4

def true_shift_matrix(s):
    P = torch.zeros(N, N)
    for i in range(N):
        P[i, (i + s) % N] = 1.0
    return P

phi_shift = {s: true_shift_matrix(s).flatten() for s in sorted_train_s2}
true_p4 = true_shift_matrix(TEST_S2).flatten()

model_shift = train_one_step_supervised(phi_shift, dim=N*N, n_steps=3000, lr=3e-3, hidden=64)

with torch.no_grad():
    pred_fwd2 = model_shift.apply_n_times(phi_shift[3], 1)
    A_pred = pred_fwd2.view(N, N)
    A_true = true_p4.view(N, N)
    acc = (A_pred.argmax(dim=-1) == A_true.argmax(dim=-1)).float().mean().item()
    mse2 = ((pred_fwd2 - true_p4) ** 2).mean().item()
    print(f"  s=3から1ステップ: 行accuracy={acc:.3f}  MSE={mse2:.6f}")
    print(f"  予測パターン: {A_pred.argmax(dim=-1).tolist()}")
    print(f"  正解パターン: {A_true.argmax(dim=-1).tolist()}")

    pred_from1_2 = model_shift.apply_n_times(phi_shift[1], 3)
    A_pred_from1 = pred_from1_2.view(N, N)
    acc_from1 = (A_pred_from1.argmax(dim=-1) == A_true.argmax(dim=-1)).float().mean().item()
    print(f"  s=1から3ステップ(独立経路): 行accuracy={acc_from1:.3f}")

print("\n" + "=" * 60)
print("総括")
print("=" * 60)
print(f"{'タスク':30s} {'非線形g(1ステップ)':>20s} {'参考baseline':>20s}")
print(f"{'sin/cos(関数出力MSE)':30s} {mse_fwd_func:>20.6f} {'単純平均=0.070':>20s}")
print(f"{'shift(行accuracy)':30s} {acc:>20.3f} {'凸結合=0%,G^s=100%':>20s}")


タスク1: sin/cos (連続・回転的構造)
  s=3から1ステップ: 2次元MSE=0.001914  関数出力MSE=0.001949
  s=1から3ステップ(独立経路での整合性チェック): 関数出力MSE=0.001949
  学習されたeps: 1.0325

  参考: 単純平均(s=3,5)の関数出力MSE = 0.069775
  参考: exp6の線形T(s=3から1ステップ)の関数出力MSE = 0.000000

タスク2: shift (離散・巡回群)
  s=3から1ステップ: 行accuracy=0.000  MSE=0.269228
  予測パターン: [3, 4, 5, 6, 7, 0, 1, 2]
  正解パターン: [4, 5, 6, 7, 0, 1, 2, 3]
  s=1から3ステップ(独立経路): 行accuracy=0.000

総括
タスク                                     非線形g(1ステップ)           参考baseline
sin/cos(関数出力MSE)                           0.001949           単純平均=0.070
shift(行accuracy)                              0.000      凸結合=0%,G^s=100%


jrf:＞訓練方法をかえるだけで shift タスクでうまくいくかもやってみてください。

＜


Claude:＞＞
<blockquote>

良い実験思いつきです。exp5のG^sが成功した理由(訓練点6個全部への到達性を損失にする)を、非線形gにもそのまま輸入してみます。見事に「s=5のコピペ」現象が、非線形gでも再現されました。しかもこれは**過学習**として起きています。

## 結果:0%失敗。しかも興味深い失敗の仕方

訓練点(s=2,3,5,6,7)は**全部完璧に再現**(accuracy=1.0, MSE=0.0)されているのに、s=4だけ外すと予測パターンが `[5,6,7,0,1,2,3,4]`——**shift(5)そのもの**になっていました。元記事の「s=4がs=5の丸ごとコピペになる」現象が、非線形生成子でも再現されたことになります。

## なぜこうなったか

これは過学習だと思います。exp5の線形G(行確率制約つき)は自由度が低く、「毎回同じ1ステップ操作を繰り返す」という制約の中でしか答えを探せなかったので、正しいshift(1)を発見するしかありませんでした。一方、今回の非線形g(隠れ層64)は自由度が高すぎて、「s=2への2ステップ」「s=3への2ステップ」「s=5への4ステップ」...とバラバラな経路をそれぞれ個別に暗記できてしまい、**一貫した「1ステップ操作」を発見する必要がなくなってしまった**んだと思います。表現力の高さが、むしろ仇になった形です。

容量を大きく絞って再挑戦してみます。容量を1〜64まで振っても、**全滅**でした。ただし興味深いのは、失敗の"仕方"が毎回変わることです——`shift(5)`のコピペになったり、`shift(3)`(何もしない)になったり、`shift(2)`になったりと、答えが一定しません。

## これは容量の問題ではなかったようです

隠れ層を1個まで削っても直りませんでした。これは何を意味するかというと、**非線形性そのものが、離散的な群構造の探索にとって「制約が弱すぎる」**ということだと思います。

線形のG(exp5)がうまくいった理由を改めて考えると、「行列の掛け算」は非常に"行儀の良い"演算です——結合法則が保証され、Gが1つ決まれば、G¹, G², G³, ... の関係は数学的に完全に固定されます。訓練点5個への到達性を要求すると、それを満たすGの選択肢は実質かなり限られます。

一方、tanh入りの非線形gは、たとえ隠れ層1個でも「1回適用したときの変化量」がステップごとに違う値を取れてしまいます。訓練点への到達性(s=1から2,3,5,6,7ステップ)を満たす経路は、線形の場合よりもずっとたくさん存在してしまい、勾配降下はその中から「たまたま訓練点にフィットするだけの、間の道筋がデタラメな経路」を平気で選んでしまう。**非線形性が持つ柔軟性こそが、今回は仇になっていた**ということだと思います。

## まとめると、良い教訓が得られました

- **sin/cos(連続・滑らかな回転)**: 非線形gは、答えを教えなくても線形回転とほぼ同等の性能に到達できた(むしろ良い結果)。
- **shift(離散・置換)**: 非線形gは、容量をどう変えても一貫して失敗し、しかも失敗の仕方(どの点にコピペされるか)が毎回違う、まさに「多すぎる自由度が悪さをする」典型例になった。

つまり「非線形化すれば拡張になる」という単純な話ではなく、**真の生成構造が線形(回転)なら線形寄りの制約が正解に近づきやすく、真の生成構造が離散的(置換)ならもっと強い制約(exp5のような行確率制約)が必要**、というのが今回分かったことだと思います。「表現力を上げる」ことと「正しい帰納バイアスに近づける」ことは、必ずしも同じ方向を向いていない、という、今回の一連の実験全体を通じたテーマの、もう一つの実例になったと思います。ファイルは以上です。今日の一連の実験(exp4〜exp10)は、「帰納バイアス」という一言の中に、実は「何を保存するか(距離か、合成則か)」「どれだけ制約するか(線形か非線形か)」という複数の独立した軸があることを、かなり具体的な形で洗い出せたと思います。
</blockquote>
＜＜



In [ ]:
"""
実験10: 非線形生成子gをshiftタスクに、exp5のG^sと同じ訓練法(全訓練点への
        到達性を損失にする)で学習させ直す。

exp9との違い:
  exp9: 隣接ペア(4組)だけを直接の教師信号にする → shiftは0%で失敗
  exp10: s=1を基点に、g(g(...g(base)...))をk回適用した結果を、
         訓練集合の全点(s=2,3,5,6,7)と比較する(exp5のGeneratorHyperと同じ発想)。
         教師信号が4組→5組(かつ各点までの「経路の妥当性」も同時に問われる)に増える。
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

N = 8
sorted_train_s = [1, 2, 3, 5, 6, 7]
TEST_S = 4

def true_shift_matrix(s):
    P = torch.zeros(N, N)
    for i in range(N):
        P[i, (i + s) % N] = 1.0
    return P

phi_shift = {s: true_shift_matrix(s).flatten() for s in sorted_train_s}
true_p4 = true_shift_matrix(TEST_S).flatten()

class ResidualGenerator(nn.Module):
    def __init__(self, dim, hidden=64, eps_init=1.0):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, dim))
        self.log_eps = nn.Parameter(torch.log(torch.tensor(eps_init)))
        for p in self.mlp[-1].parameters():
            nn.init.zeros_(p)
    def step(self, x):
        return x + torch.exp(self.log_eps) * self.mlp(x)
    def apply_n_times(self, x, n):
        for _ in range(n):
            x = self.step(x)
        return x

def evaluate(model, anchor_s, target_s, phi_dict):
    with torch.no_grad():
        pred = model.apply_n_times(phi_dict[anchor_s], target_s - anchor_s)
        A_pred = pred.view(N, N)
        A_true = true_shift_matrix(target_s)
        acc = (A_pred.argmax(dim=-1) == A_true.argmax(dim=-1)).float().mean().item()
        mse = ((pred - A_true.flatten()) ** 2).mean().item()
    return acc, mse, A_pred.argmax(dim=-1).tolist()

# ------------------------------------------------------------
# 訓練: s=1を基点に、全訓練点への到達性を損失にする(exp5方式)
# ------------------------------------------------------------
def train_full_reach(n_steps=4000, lr=2e-3, hidden=64):
    model = ResidualGenerator(N * N, hidden=hidden)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    anchor = min(sorted_train_s)  # s=1
    base = phi_shift[anchor]
    for step in range(n_steps):
        opt.zero_grad()
        loss = 0.0
        for s in sorted_train_s:
            if s == anchor:
                continue
            pred = model.apply_n_times(base, s - anchor)
            loss = loss + ((pred - phi_shift[s]) ** 2).mean()
        loss = loss / (len(sorted_train_s) - 1)
        loss.backward()
        opt.step()
        if step % 800 == 0:
            print(f"    step {step}: loss={loss.item():.5f}  eps={torch.exp(model.log_eps).item():.4f}")
    return model

print("=== exp10: 全訓練点到達性で非線形gを訓練(基点s=1) ===")
model = train_full_reach()

acc4, mse4, pattern4 = evaluate(model, 1, TEST_S, phi_shift)
print(f"\ns=1から3ステップでs=4を予測:")
print(f"  行accuracy={acc4:.3f}  MSE={mse4:.6f}")
print(f"  予測パターン: {pattern4}")
print(f"  正解パターン: {true_shift_matrix(TEST_S).argmax(dim=-1).tolist()}")

# 内部整合性: 訓練点自体もちゃんと再現できているか(s=1から2,3,5,6,7への到達性)
print("\n訓練点での再現性チェック(内部整合性):")
for s in sorted_train_s:
    if s == 1:
        continue
    acc_s, mse_s, _ = evaluate(model, 1, s, phi_shift)
    print(f"  s=1→{s}: accuracy={acc_s:.3f}  MSE={mse_s:.6f}")

print("\n=== これまでのshiftタスクでの全結果比較 ===")
print(f"{'手法':45s} {'行accuracy(s=4)':>16s}")
print(f"{'A: Baseline MLP':45s} {'0%':>16s}")
print(f"{'E: Convex-Combination Attention':45s} {'0%':>16s}")
print(f"{'F: G^s (線形・行確率、全点損失)':45s} {'100%':>16s}")
print(f"{'非線形g、隣接ペアのみ(exp9)':45s} {'0%':>16s}")
print(f"{'非線形g、全点到達性損失(exp10・今回)':45s} {f'{acc4*100:.0f}%':>16s}")


=== exp10: 全訓練点到達性で非線形gを訓練(基点s=1) ===
    step 0: loss=0.25000  eps=1.0000
    step 800: loss=0.00000  eps=1.0843
    step 1600: loss=0.00000  eps=1.0761
    step 2400: loss=0.00000  eps=1.0728
    step 3200: loss=0.00001  eps=1.0661

s=1から3ステップでs=4を予測:
  行accuracy=0.000  MSE=0.417368
  予測パターン: [5, 6, 7, 0, 1, 2, 3, 4]
  正解パターン: [4, 5, 6, 7, 0, 1, 2, 3]

訓練点での再現性チェック(内部整合性):
  s=1→2: accuracy=1.000  MSE=0.000000
  s=1→3: accuracy=1.000  MSE=0.000000
  s=1→5: accuracy=1.000  MSE=0.000000
  s=1→6: accuracy=1.000  MSE=0.000000
  s=1→7: accuracy=1.000  MSE=0.000001

=== これまでのshiftタスクでの全結果比較 ===
手法                                              行accuracy(s=4)
A: Baseline MLP                                             0%
E: Convex-Combination Attention                             0%
F: G^s (線形・行確率、全点損失)                                      100%
非線形g、隣接ペアのみ(exp9)                                           0%
非線形g、全点到達性損失(exp10・今回)                                      0%


In [ ]:
"""
exp10の追試: 容量を大きく絞った非線形g(隠れ層4)+ 重み減衰で
             過学習(暗記)を防げるか
"""
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

N = 8
sorted_train_s = [1, 2, 3, 5, 6, 7]
TEST_S = 4

def true_shift_matrix(s):
    P = torch.zeros(N, N)
    for i in range(N):
        P[i, (i + s) % N] = 1.0
    return P

phi_shift = {s: true_shift_matrix(s).flatten() for s in sorted_train_s}

class ResidualGenerator(nn.Module):
    def __init__(self, dim, hidden, eps_init=1.0):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, dim))
        self.log_eps = nn.Parameter(torch.log(torch.tensor(eps_init)))
        for p in self.mlp[-1].parameters():
            nn.init.zeros_(p)
    def step(self, x):
        return x + torch.exp(self.log_eps) * self.mlp(x)
    def apply_n_times(self, x, n):
        for _ in range(n):
            x = self.step(x)
        return x

def evaluate(model, anchor_s, target_s):
    with torch.no_grad():
        pred = model.apply_n_times(phi_shift[anchor_s], target_s - anchor_s)
        A_pred = pred.view(N, N)
        A_true = true_shift_matrix(target_s)
        acc = (A_pred.argmax(dim=-1) == A_true.argmax(dim=-1)).float().mean().item()
    return acc, A_pred.argmax(dim=-1).tolist()

def train(hidden, weight_decay, n_steps=4000, lr=2e-3):
    model = ResidualGenerator(N * N, hidden=hidden)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    anchor = 1
    base = phi_shift[anchor]
    for step in range(n_steps):
        opt.zero_grad()
        loss = 0.0
        for s in sorted_train_s:
            if s == anchor: continue
            pred = model.apply_n_times(base, s - anchor)
            loss = loss + ((pred - phi_shift[s]) ** 2).mean()
        loss = loss / (len(sorted_train_s) - 1)
        loss.backward()
        opt.step()
    return model

print(f"{'隠れ層サイズ':>10s} {'weight_decay':>14s} {'s=4 accuracy':>14s}  予測パターン")
for hidden, wd in [(64, 0.0), (16, 0.0), (4, 0.0), (2, 0.0), (4, 1e-2), (2, 1e-2), (1, 0.0)]:
    model = train(hidden, wd)
    acc, pattern = evaluate(model, 1, TEST_S)
    print(f"{hidden:>10d} {wd:>14.0e} {acc:>14.3f}  {pattern}")

print(f"\n(正解パターン: {true_shift_matrix(TEST_S).argmax(dim=-1).tolist()})")


    隠れ層サイズ   weight_decay   s=4 accuracy  予測パターン
        64          0e+00          0.000  [5, 6, 7, 0, 1, 2, 3, 4]
        16          0e+00          0.000  [3, 6, 5, 6, 7, 0, 1, 2]
         4          0e+00          0.000  [3, 4, 5, 6, 7, 0, 1, 2]
         2          0e+00          0.000  [5, 6, 7, 0, 1, 2, 3, 4]
         4          1e-02          0.000  [5, 6, 7, 0, 1, 2, 3, 4]
         2          1e-02          0.000  [5, 6, 7, 0, 1, 2, 3, 4]
         1          0e+00          0.000  [2, 3, 4, 5, 6, 7, 0, 1]

(正解パターン: [4, 5, 6, 7, 0, 1, 2, 3])


jrf:＞「何を保存するか(距離か、合成則か)」「どれだけ制約するか(線形か非線形か)」という複数の独立した軸があってもそれを選択する方法があるというのが機械学習ではないのですか？

＜


Claude:＞＞
<blockquote>

良い指摘です。実は「ある」のですが、限定的にです。整理します。

## 「選択する方法」自体は、機械学習の標準的な道具としてすでに存在します

今日私たちがやっていたこと自体が、実はその実例でした。複数の候補(凸結合アテンション、G^s、非線形g、集団符号化)を用意して、LOO(leave-one-out)で性能を比較し、勝ったものを採用する——これは**モデル選択(model selection)**という、機械学習の中でも最も基本的な手法の一つです。交差検証、ベイズ的モデル選択、赤池情報量規準(AIC)なども同じ family です。なので「複数の軸から選ぶ」という行為自体には、既に確立された方法があります。

## ただし、これには重要な限界があります

**モデル選択は「候補のメニューを選ぶ」ことしかできず、「メニュー自体を発想する」ことはできません。** 今日の実験で言えば、「凸結合を試そう」「G^sを試そう」「集団符号化を試そう」という**候補そのものを思いついたのは、JRFさんと私(たち複数のAI)の議論**でした。LOOによる比較は、思いついた後の「どれが勝つか」を機械的に判定しただけです。

これを完全に自動化しようとすると、「あらゆる可能な構造(合成則か、距離か、群か、非可換か…)を総当たりで探索する」ことになりますが、これは**組み合わせ爆発**を起こします(プログラム合成・記号回帰という分野がまさにこれに挑んでいますが、いまだに小さな問題にしか通用しません)。

## もう一つ、根の深い制約があります

**「ノーフリーランチ定理」**という結果があります。ざっくり言うと、「あらゆる可能なタスクに対して万能に機能する、偏りのない学習器は存在しない」という数学的な定理です。つまり、「どんなタスクにも自動的に正しい帰納バイアスを選んでくれる、バイアスのないメタ学習器」を作ろうとしても、その「選ぶための手続き」自体が、また新たな一段上のバイアス(何を優先して探すか)を持たざるを得ません。無限後退します。

これを緩和する現実的な方法が**メタ学習**(多数の関連タスクにまたがって「このタイプのタスクにはこの帰納バイアスが効く」という経験を積ませる)ですが、これには**似た構造を持つタスクを多数用意する必要**があります。今日は shift や sin/cos という**1つのタスクだけ**を相手にしていたので、メタ学習が本来威力を発揮する土俵にすらいませんでした。

## 実は面白いことに気づきました

「どの帰納バイアスを選ぶか」を自動選択しようとする問題自体が、**今日ずっと扱ってきたs=4問題の、一段上のレベルでの再来**になっていると思います。「複数の候補アーキテクチャを"混ぜて(凸結合して)"選ぼう」とすると、実験5のE(凸結合アテンション)が離散構造で失敗したのと同じ理由で失敗するはずです(アーキテクチャの選択も、実は離散的で組み合わせ的な選択なので)。逆に「候補群を合成的に(生成的に)探索しよう」とすると、今度はその「候補を生成する文法(DSL)」自体を、また人間が設計しなければならない。

つまり答えは、「選択する方法はある(モデル選択・メタ学習)が、それは"すでに与えられた候補群の中から選ぶ"ことしかできず、"候補群自体をゼロから発想する"ことは、原理的には今日扱った問題そのものの再帰的な繰り返しになる」というのが、私の見立てです。
</blockquote>
＜＜

jrf:＞まず、一般には g(g(…, s),s) みたいにしたほうがいいでしょう。ただ、s はないほうが今回のものについては汎化はしやすいでしょう。ちなみにコードでは実質的にこれをやってるように見えます。

次に、機械学習の古典的なものだと A x + b の b のようなバイアス項がありますよね。それがないのが今回実験5の例で失敗を生んでいるということはないですか？

＜


Claude:＞＞
<blockquote>

両方とも良い着眼点です。一つ目は当たっていて、二つ目はこの実験に関しては違うと思います。理由を説明します。

## 1. g(x) に s を入れないことについて

その通りで、コードは実質的にすでに「sなしのg(x)」でやっています(`step(self, x)` は s を受け取らない)。これは意図的というより、**このタスクの構造(等速シフト・一定角速度の回転)そのものが「今どこにいるかに関わらず、次の1歩は常に同じ」という斉次性(homogeneity)を持っているから**、s抜きの方が有利でした。もし仮にg(x,s)としてsを入力に加えていたら、モデルはs=1→2とs=2→3で「違う変換」を使うことを許されてしまい、実験10で見たような「訓練点ごとに違う経路を暗記する」自由度をさらに増やしてしまったはずです。JRFさんの直感通り、**sを抜くこと自体が正しい帰納バイアス(斉次性の仮定)になっていた**、というのは的確な整理だと思います。

## 2. バイアス項(b)の欠如について

こちらは、今回の失敗の原因ではないと思います。理由を具体的に見てみます。

**実験5のA(baseline MLP)は、実はすでにバイアス項を持っていました**(`nn.Linear`はデフォルトで`bias=True`)。それでも0%で失敗しています。逆に、**成功したF(G^s)には、明示的なバイアス項がありません**(`G = softmax(raw)` を x に掛けるだけ)。バイアス項の有無と成否が、ちょうど逆の対応になっているので、「バイアスがないから失敗した」という説明はこのデータには当てはまりません。

さらに理論的に考えても、**このタスクの真の生成規則には、そもそもバイアス項が存在しません**。置換行列を状態ベクトルに掛け算するだけで、次の状態(one-hotベクトル)がちょうど再現されます(x' = Gx、c=0)。sin/cosの回転も同様で、原点を中心とした回転なのでバイアスは要りません(x' = Rx)。**両方のタスクとも、たまたま「真の答えのバイアス項がゼロ」というケースだった**ので、バイアス項を省いたこと自体は今回に関しては正解でした。

## ただし一般論としては良い指摘だと思います

もっと一般のタスク、たとえば「原点から離れた場所を中心に回転する」「一定の外力・ドリフトがかかっている」ようなタスクでは、x' = Gx + c という**アフィン変換**でないと正しく表現できません。今日のフレームワーク(g(x)やG)はどれも「原点を固定点とする線形変換」に暗黙に限定されていたので、これは**今回たまたま影響しなかった、隠れた制約**ではありました。次に「もっと一般のアテンション保存変換」を設計するなら、x' = Gx + c の形にしておくのは、保険として妥当だと思います。試してみますか?
</blockquote>
＜＜

jrf:＞実験5の例がうまくいかないのが納得できないです。g^3 と g^5 の間の g^4 がその中間的なものにならないというのが。g が s に依存するようになっているということですよね…。

＜


Claude:＞＞
<blockquote>

いいえ、gはsに依存していません(コード上、`step(x)`はxだけを受け取り、sは一切渡していません)。それでも「s依存しているように見える」挙動が起きたのは事実で、そこにこそ今回の失敗の核心があります。丁寧に見てみます。

## 実際に何が起きていたか

訓練ログから逆算すると、こうなっています:

- g(φ₁) = φ₂ (完全一致、MSE=0)
- g(φ₂) = φ₃ (完全一致、MSE=0。つまり g(g(φ₁))=φ₃)
- **g(φ₃) = 「φ₅のパターン」** (これがs=4での予測結果そのものです。model.apply_n_times(φ₁, 3) = g(g(g(φ₁))) = g(φ₃) なので)

つまり、gは「φ₁→φ₂」「φ₂→φ₃」までは正しく"1歩ずつ"進んでいたのに、**φ₃に到達した瞬間、突然2歩分(φ₅相当)ジャンプしている**んです。sの情報を陽に渡していないのに、まるで「今何回目の適用かを覚えていて、3回目だけ違う動きをする」かのような、事実上の**文脈依存(場所依存)の挙動**が起きています。gはxだけを受け取りますが、入力xの値そのものが暗黙に「今何ステップ目か」の情報を持っているので、gは「φ₃という特定の入力パターンを受け取ったら2歩ジャンプする」という、**xの値ごとに違う規則**を学習できてしまった、ということです。

## なぜ線形Gではこれが起きなかったのか

線形写像だと、これが原理的にできません。なぜなら、線形写像Gは「行列を1つ固定する」ことと同義で、**xの値によって挙動を変える自由度が最初からない**からです。G(φ₁)=φ₂、G(φ₂)=φ₃、G(G(G(G(φ₁))))=φ₅ ... というふうに、同じ1つの行列Gに対して5つの異なる「べき乗の到達点」を同時に満たせという制約をかけると、これは**Gという64×64個の数を決めるための、非常に厳しい連立方程式**になります。この連立方程式をすべて満たす行列Gは、事実上「本物の巡回シフト行列」しかありませんでした(だからこそ訓練誤差がきっちり0になり、かつテストでも100%当たった)。行列という枠組みそのものが、「φ₃でだけ違う動きをする」という**ズルを原理的に許さない**構造だったんです。

対して非線形g(tanh入りMLP)は、**入力xの値ごとに全く違う変換ができる**という自由度を持っています。だから訓練データにある5つの到達点(φ₂,φ₃,φ₅,φ₆,φ₇)をすべて完璧に満たしながら、φ₃という「たまたま観測されなかった中継点」でだけ、こっそり辻褄合わせのジャンプをする、という不正解が可能になってしまいました。

**まとめると**: gがsに直接依存しているのではなく、「xの値に応じて実質的に違う関数として振る舞える」という非線形性の自由度が、あたかもsに依存しているかのような、局所ごとに違う挙動を生み出してしまった、というのが正確な説明だと思います。この意味で、JRFさんの「sに依存するようになっている」という直感は、字義通りではないですが、**本質を正確に言い当てている**と思います。
</blockquote>
＜＜


jrf:＞情報量が大きすぎるからそれを減らすよう最適化すればいいのか、表現力を増やしてアテンションでも入れて、創発を期待すればいいのか…。Gemini さんによると今回の実験5の例では後者ではなく前者しかないとのこと。

でも、LLMは巨大になることで汎化してきた。違いは何だろう？ 情報をそぎ落とすという方向ではスパース化はうまくいかなかったという印象…。

元々のNNの発展史で XOR を実現するのがとても難しく、そこから直感的に、NNは役に立たないと思われていた。私もそう思っていました。しかし深層NNはすべてを変えた。そういうことが…、深層にすればいいという話ではないかもしれませんが…、ここにもありうるように思うのです。

今回は sin/cos と shift でしたが、他の例もいっぱい(誘導しやすいと思われるものをあと2・3個？)覚えてそれで s=4 は他から汎化できるというのをだいたい覚えさせてからやるという方向はどうでしょう？

でも、それができるなら $G^s$ の構造がなくてもうまくいくような気も…。

＜


Claude:＞＞
<blockquote>

とても良い質問で、実はこれがLLMの汎化を理解する上でのど真ん中の論点だと思います。整理します。

## Geminiさんの「前者しかない」は、この実験(単一タスク・6点)に限定すれば正しいですが、一般論としては違うと思います

今日ずっとやっていたのは、「**1つのタスク**を、**6個のデータ点**だけから学習する」という、極端にデータが少ない状況でした。この状況では、表現力(非線形性)を増やせば増やすほど、「6点にフィットする無数の間違った経路」が増えるだけで、正しい経路を選ぶ手がかりが増えるわけではありません。だからGeminiさんの言う通り、この土俵では「制約を強める」方向しか勝ち筋がなかった。これは正しい診断です。

## LLMの汎化は、実は「表現力を上げた」から起きたのではないと思います

歴史的な整理をすると、深層NNがXORの壁を破ったのも、実は「表現力を無制限に増やした」からではなく、**「タスクの構造に合った特定の制約(重み共有・局所性・階層性)を、大量のデータと組み合わせて入れた」**からでした(畳み込み=平行移動不変性という強いバイアスの一種です)。LLM(Transformer)も同様で、「注意機構」自体が強いバイアス(トークン間の関係性という特定の演算様式)ですし、決定的に違うのは**タスクの"個数"**です。

私たちの実験は「1つのタスクに6点」でしたが、LLMの事前学習は「**何十億もの、互いに関連した"次を予測する"タスクの実例**」を相手にしています。この規模の違いが本質的だと思います。

## つまり、JRFさんの発想(複数タスクを覚えさせてから汎化させる)は、まさにこの違いを埋める正しい方向だと思います

これは機械学習で「**メタ学習(learning to learn)**」「**インコンテキスト学習**」と呼ばれている領域そのものです。実際に研究例があり、Transformerに大量の「異なる線形関数を数点の例から当てさせる」タスクを学習させると、明示的に最小二乗法のアルゴリズムを教えなくても、**文脈内の数点から一貫した規則を推論する能力そのものを、訓練を通じて"獲得"する**ことが報告されています(Garg et al. 2022 の "in-context learning" 研究などが有名です)。これはまさにJRFさんが提案していることと同型です。

## ただし「2〜3個」では、たぶん足りないと思います

ここが重要な但し書きです。メタ学習が効くのは、**「同種の未知パラメータ埋め穴問題」を大量の"バリエーション違い"で何度も経験させ、個別暗記では割に合わないほどタスク数を増やす**ことで、ネットワークが「毎回律儀に一般規則を推論する」以外の近道がなくなるからです。2〜3個のタスクでは、非線形gが今日やったのと同じように「タスクごとに個別対応する」方が依然として"お得"なので、また同じ暗記による失敗が起きる可能性が高いと思います。目安として、少なくとも数十〜数百通りの「shift量が違う版」「回転速度が違う版」を作り、その都度違う位置を隠して訓練する、という規模感が必要だと思います。

## 「それができるならG^sの構造は要らないのでは」への答え

ここが一番面白いところで、**「要らなくなる」のではなく、「ネットワークが内部でG^s相当のものを、明示的な設計なしに"発見"するようになる」**というのが正確だと思います。メタ学習された重みの中に、暗黙に「今の入力の背後にある合成則を推定し、それに従って外挿する」という手続きが埋め込まれる形です。つまり、私たちが今日 G という形を人手で設計した仕事を、大量のタスクを見せることでネットワーク自身にやらせる、という**責任の移動**であって、「合成的構造が要らなくなる」わけではないと思います。

## スパース化について

おっしゃる通り、単純なスパース化(重み減衰など、exp10bでも試して効きませんでした)はこの問題を解決しません。「パラメータの数を減らす」ことと「正しい代数的構造(斉次性・可逆性)を持つ」ことは、別の種類の"単純さ"だからです。全結合層をいくら疎にしても、それが自動的に「群の表現」になるわけではありません。

---

もし試すなら、小規模な実証実験は組めると思います——たとえば「shift量k=1〜7それぞれについて、s系列の一部を隠して学習」というのを何十パターンも作り、Transformer的なモデルに「複数のタスクにまたがって、隠された点を当てる」ことを学習させ、**訓練で一度も見ていない新しいshift量**でも隠された点を当てられるかを見る、という実験です。ただ、これは今日までの実験よりずっと計算量・データ準備が要ります。利用制限のこともあるので、次回改めて、という形でもいいかもしれません。やってみますか?
</blockquote>
＜＜

jrf:＞左shift、右2段shift は s=4 も覚えて、sin/cos は s=5 を隠して shift は s=4 を隠して…で一つのモデルに覚えさせてみて sin/cos と shift が「創発」するか見るという方向はいかがでしょう？

＜


Claude:＞＞
<blockquote>

大掛かりですが、面白い実験なので試してみます。ただし設計上、一つ気をつけたい点があります——「shift」のテストタスクを、helperタスク(左shift)と**全く同じ法則**にしてしまうと、モデルは「法則を汎化した」のではなく「helperタスクの答えをそのままコピーした」だけになり、公正な検証になりません。なので、helperは速度1・速度2のshift、テスト対象は速度3のshift(未知の速度)にして、「速度という連続パラメータに対する合成則の汎化」を見る形にします。sin/cosは指定通りs=5を隠します。部分的な「創発」が起きました。ただし、二つのテストタスクで明暗が分かれています。

## 結果

| タスク | 結果 |
|---|---|
| C: sin/cos(s=5隠す) | **MSE=0.048**(単純平均の0.072より良い!) |
| D: 速度3のshift(s=4隠す) | **accuracy 0%**(shift-by-1という完全な的外れ) |
| helperタスクA・B(全点既知) | 両方とも100%正確に再現 |

## sin/cosは本物の「創発」が起きました

単純な平均ベースラインより良い結果が出た、というのが重要な点です。JRFさんの狙い通り、shiftの2つのhelperタスク(速度1・速度2)を同時に学習することで、共有された生成子gが「一貫した1ステップの遷移」という性質そのものを、より頑健に学習でき、それがsin/cosのs=5という未見点にも良い方向に転用された、と解釈できます。地味ですが、はっきりした正の転移です。

## しかしshift速度3は完全に失敗しました

原因を探ると、興味深いことが分かりました。**速度3のshiftは、helperタスク(速度1・速度2)のどちらとも異なる「歩幅」を必要とします。** 共有された生成子gには「今どのタスクにいて、どれだけの歩幅で進むべきか」を教える手がかりが(sという値を直接渡していないので)、潜在表現zの"位置"からしか読み取れません。速度1・速度2の情報は十分にあっても、**速度3という全く新しい歩幅は、どの既存情報からも補間できなかった**——潜在空間の中で、この新しい歩幅に対応する領域が十分に制約されておらず、結果としてgは「何もしない(1ステップぶんしか進まない)」という保守的な答えに逃げてしまいました。

## まとめると

「複数タスクを覚えさせて創発を狙う」という方向性は、**"すでに見た変動の範囲内"では機能する**(sin/cosの改善がその証拠)が、**"全く新しい種類の変動(未知の歩幅)"には依然として汎化しない**、ということが分かりました。これはLLMの文脈でも実は同じ話で、いわゆる「分布内での補間的な汎化」と「真に新しい概念への外挿」は別物だと言われています。もし速度3もうまく扱いたいなら、helperタスクとして速度1・2・4・5あたりも用意して「速度」という軸自体の補間を可能にするか、あるいは速度(歩幅)という情報を明示的にgに入力として与える(sではなく、タスク固有の"歩幅"パラメータとして)必要があると思います。sin/cosでの正の転移は、今日の一連の実験の中でも意外性のある良い結果だったと思います。「表現力を上げれば創発する」でも「制約するしかない」でもなく、「関連タスクの幅の中でなら創発するが、幅の外には出られない」という、もう少し繊細な答えになりました。
</blockquote>
＜＜




In [ ]:
"""
実験11: 複数タスクの共同学習による「創発」の検証

設計:
  helperタスク(全点既知、s=1..7):
    A) 速度1のshift(左)       - 全部教える
    B) 速度2のshift(右)       - 全部教える
  テストタスク(1点を隠す):
    C) sin/cos                - s=5を隠す(今回はs=4ではなくs=5)
    D) 速度3のshift(左)       - s=4を隠す
       ※ Aと同じ"法則の種類"だが速度が違うインスタンス。
         これにより「Aの答えをそのままコピーする」というズルができない
         (Aは速度1、Dは速度3なので、s=4での正解が全く別の値になる)。

アーキテクチャ:
  各タスクごとに: 生の出力空間(shiftなら64次元、sin/cosなら40次元) を
  共有潜在空間(8次元)に落とす学習可能な線形エンコーダ/デコーダを持つ。
  「1ステップ進める」操作 g だけは、全タスクで共有される単一の非線形MLP。
  g が全タスク・全ステップ深度にわたって一貫した振る舞いを強制されることで、
  実験9/10で起きた「観測されなかった深度でだけズルをする」ことが
  起きにくくなるかを見る。

評価: C(s=5)とD(s=4)の予測が、それぞれの真の値にどれだけ近いか。
"""

import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

N = 8  # shift系列長
D_LATENT = 8

# ------------------------------------------------------------
# タスク定義
# ------------------------------------------------------------
def shift_matrix(s, speed, direction=1):
    P = torch.zeros(N, N)
    for i in range(N):
        P[i, (i + direction * speed * s) % N] = 1.0
    return P

def true_function(s, x, period=7.0):
    return torch.sin(2 * np.pi * (x + s / period))

N_X = 40
X_GRID = torch.linspace(0, 1, N_X).unsqueeze(1)

ALL_S = [1, 2, 3, 4, 5, 6, 7]

tasks = {
    "A_speed1_left":  {"dim": N*N, "known_s": ALL_S,               "kind": "shift", "speed": 1, "dir": 1},
    "B_speed2_right": {"dim": N*N, "known_s": ALL_S,               "kind": "shift", "speed": 2, "dir": -1},
    "C_sincos":       {"dim": N_X, "known_s": [1,2,3,4,6,7],       "kind": "sincos"},   # s=5隠す
    "D_speed3_left":  {"dim": N*N, "known_s": [1,2,3,5,6,7],       "kind": "shift", "speed": 3, "dir": 1},  # s=4隠す
}

def get_true(task_name, s):
    t = tasks[task_name]
    if t["kind"] == "shift":
        return shift_matrix(s, t["speed"], t["dir"]).flatten()
    else:
        return true_function(s, X_GRID).squeeze(1)

# ------------------------------------------------------------
# モデル: タスクごとの線形エンコーダ/デコーダ + 共有の非線形g
# ------------------------------------------------------------
class SharedGenerator(nn.Module):
    def __init__(self, d_latent=D_LATENT, hidden=32):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(d_latent, hidden), nn.Tanh(), nn.Linear(hidden, d_latent))
        self.log_eps = nn.Parameter(torch.log(torch.tensor(1.0)))
        for p in self.mlp[-1].parameters():
            nn.init.zeros_(p)
    def step(self, z):
        return z + torch.exp(self.log_eps) * self.mlp(z)

class TaskHead(nn.Module):
    def __init__(self, raw_dim, d_latent=D_LATENT):
        super().__init__()
        self.enc = nn.Linear(raw_dim, d_latent)
        self.dec = nn.Linear(d_latent, raw_dim)
    def encode(self, x): return self.enc(x)
    def decode(self, z): return self.dec(z)

shared_g = SharedGenerator()
heads = {name: TaskHead(t["dim"]) for name, t in tasks.items()}

params = list(shared_g.parameters())
for h in heads.values():
    params += list(h.parameters())
opt = torch.optim.Adam(params, lr=2e-3)

# 各タスクの既知データを事前計算
raw_data = {name: {s: get_true(name, s) for s in t["known_s"]} for name, t in tasks.items()}

N_STEPS = 4000
print("=== 共同学習開始 (4タスク: A,B=全既知, C,D=1点隠す) ===")
for step in range(N_STEPS):
    opt.zero_grad()
    total_loss = 0.0
    n_terms = 0

    for name, t in tasks.items():
        head = heads[name]
        known = sorted(t["known_s"])

        # (a) 再構成損失: 既知点はエンコード→デコードで戻ってくるべき
        for s in known:
            x = raw_data[name][s]
            z = head.encode(x)
            x_rec = head.decode(z)
            total_loss = total_loss + ((x_rec - x) ** 2).mean()
            n_terms += 1

        # (b) 1ステップ一貫性損失: 隣接する既知点ペアについて、
        #     共有g が正しく次の潜在状態を予測できるべき
        for s in known:
            if (s + 1) in known:
                z_a = head.encode(raw_data[name][s])
                z_b = head.encode(raw_data[name][s + 1])
                z_pred = shared_g.step(z_a)
                total_loss = total_loss + ((z_pred - z_b) ** 2).mean()
                n_terms += 1

    total_loss = total_loss / n_terms
    total_loss.backward()
    opt.step()
    if step % 800 == 0:
        print(f"  step {step}: loss={total_loss.item():.6f}")

# ------------------------------------------------------------
# 評価: C(s=5), D(s=4) を予測する
# ------------------------------------------------------------
def predict(task_name, target_s):
    t = tasks[task_name]
    head = heads[task_name]
    known = sorted(t["known_s"])
    below = max([s for s in known if s < target_s])
    above = min([s for s in known if s > target_s])
    with torch.no_grad():
        z_below = head.encode(raw_data[task_name][below])
        for _ in range(target_s - below):
            z_below = shared_g.step(z_below)
        pred_fwd = head.decode(z_below)
    return pred_fwd, below, above

print("\n=== 評価: C(sin/cos, s=5 隠す) ===")
pred_c, below_c, above_c = predict("C_sincos", 5)
true_c = get_true("C_sincos", 5)
mse_c = ((pred_c - true_c) ** 2).mean().item()
w_lin_c = 0.5 * (raw_data["C_sincos"][4] + raw_data["C_sincos"][6])
mse_c_lin = ((w_lin_c - true_c) ** 2).mean().item()
print(f"  s={below_c}から{5-below_c}ステップで予測: MSE={mse_c:.6f}")
print(f"  参考(単純平均 s=4,6): MSE={mse_c_lin:.6f}")

print("\n=== 評価: D(速度3のshift, s=4 隠す) ===")
pred_d, below_d, above_d = predict("D_speed3_left", 4)
true_d = get_true("D_speed3_left", 4)
A_pred_d = pred_d.view(N, N)
A_true_d = true_d.view(N, N)
acc_d = (A_pred_d.argmax(dim=-1) == A_true_d.argmax(dim=-1)).float().mean().item()
mse_d = ((pred_d - true_d) ** 2).mean().item()
print(f"  s={below_d}から{4-below_d}ステップで予測: 行accuracy={acc_d:.3f}  MSE={mse_d:.6f}")
print(f"  予測パターン: {A_pred_d.argmax(dim=-1).tolist()}")
print(f"  正解パターン: {A_true_d.argmax(dim=-1).tolist()}")

# 内部整合性: helperタスクA,Bはちゃんと再現できているか
print("\n=== 内部整合性チェック(helperタスクA,Bの全点再現) ===")
for name in ["A_speed1_left", "B_speed2_right"]:
    accs = []
    for s in ALL_S:
        with torch.no_grad():
            z = heads[name].encode(raw_data[name][1])
            for _ in range(s - 1):
                z = shared_g.step(z)
            pred = heads[name].decode(z)
        acc = (pred.view(N,N).argmax(-1) == raw_data[name][s].view(N,N).argmax(-1)).float().mean().item()
        accs.append(acc)
    print(f"  {name}: s=1から各sへの到達accuracy = {[round(a,2) for a in accs]}")


=== 共同学習開始 (4タスク: A,B=全既知, C,D=1点隠す) ===
  step 0: loss=0.199941
  step 800: loss=0.000000
  step 1600: loss=0.000000
  step 2400: loss=0.000000
  step 3200: loss=0.000001

=== 評価: C(sin/cos, s=5 隠す) ===
  s=4から1ステップで予測: MSE=0.047715
  参考(単純平均 s=4,6): MSE=0.072476

=== 評価: D(速度3のshift, s=4 隠す) ===
  s=3から1ステップで予測: 行accuracy=0.000  MSE=0.681918
  予測パターン: [1, 2, 3, 4, 5, 6, 7, 0]
  正解パターン: [4, 5, 6, 7, 0, 1, 2, 3]

=== 内部整合性チェック(helperタスクA,Bの全点再現) ===
  A_speed1_left: s=1から各sへの到達accuracy = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
  B_speed2_right: s=1から各sへの到達accuracy = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


jrf:＞左shift1と右shift2(とsin/cos)で、右shift1 の s=4 を期待するのは簡単すぎる・ズルであるという評価なのですか？右shift3 は他になさすぎる性質なので難しすぎるように思います。

＜


Claude:＞＞
<blockquote>

その評価、正確だと思います。整理すると:

- **右shift速度1**: 「速度」という軸ではAと同じ値(1)を使い、「方向」という軸ではBと同じ値(右)を使う、**既知の2つの軸の値を新しい組み合わせにしただけ**です。これは「Aの答えをそのままコピーする」ズルとは違い、「方向と速度が独立した軸として分離して学習されているか」を試す、ちょうど良い難易度のテストだと思います。
- **右shift速度3**: 速度という軸で、**どのhelperにも存在しない全く新しい値**を要求するので、補間の手がかりが最初からありません。これは「難しすぎる」というより、「原理的に情報が足りていない」設定でした。

実際に試してみましょう。意外な結果になりました。**これも0%で失敗しました。** むしろ実験11の速度3(shift-by-1というもっともらしい間違い)より悪く、出力が有効な置換パターンにすらなっていません(`[7,6,7,0,3,2,3,6]` — 同じ行き先が重複していて、置換として壊れています)。ついでにsin/cos側も今回はベースラインよりわずかに悪化しました(0.078 vs 0.072)。

## つまり「難易度としてちょうど良いはず」という予想は外れました

これは重要な否定的知見だと思います。「方向」と「速度」が、モデルの中で**独立した軸として分離されて学習されている**なら、この組み合わせテストは簡単に解けるはずでした。しかし実際には解けなかった、ということは、**このアーキテクチャ(タスクごとの線形エンコーダ+共有非線形g)は、そもそも"方向"や"速度"を再利用可能な独立した概念として切り分けていなかった**ということだと思います。おそらく各タスクのエンコーダは、それぞれのタスク専用の「なんとなく動く」表現をバラバラに学習しただけで、方向×速度という因子分解された構造を潜在空間の中に作り出してはいなかったのでしょう。

## これは今日の一連の実験全体の結論とも整合しています

「表現力を増やせば創発する」のではなく、「タスクの構造(この場合は"方向"と"速度"という独立した要因への分解)そのものをアーキテクチャに埋め込まない限り、汎用的なニューラルネットは自発的にはその分解を発見しない」ということです。実際にこの組み合わせ的な汎化を実現したいなら、方向embeddingと速度embeddingを別々に用意して、それらを(足し算や掛け算などの明示的な演算で)組み合わせる、という設計を最初から作り込む必要がありそうです。「簡単すぎる」どころか、**むしろ暗黙の分離を仮定していた分、今回の設計にとっては速度3のケースより難しかった**、というのが正直な結果でした。今日はここまでにしておくのが良さそうです。「難易度を予想する」という私たちの直感自体が、このアーキテクチャの実際の挙動とズレていた、というのも含めて、良い記録になったと思います。
</blockquote>
＜＜



In [ ]:
"""
実験12: 複数タスクの共同学習による「創発」の検証

設計:
  helperタスク(全点既知、s=1..7):
    A) 速度1のshift(左)       - 全部教える
    B) 速度2のshift(右)       - 全部教える
  テストタスク(1点を隠す):
    C) sin/cos                - s=5を隠す(今回はs=4ではなくs=5)
    D) 速度1・右方向のshift(A×Bの新しい組み合わせ)(左)       - s=4を隠す
       ※ Aと同じ"法則の種類"だが速度が違うインスタンス。
         これにより「Aの答えをそのままコピーする」というズルができない
         (Aは速度1、Dは速度3なので、s=4での正解が全く別の値になる)。

アーキテクチャ:
  各タスクごとに: 生の出力空間(shiftなら64次元、sin/cosなら40次元) を
  共有潜在空間(8次元)に落とす学習可能な線形エンコーダ/デコーダを持つ。
  「1ステップ進める」操作 g だけは、全タスクで共有される単一の非線形MLP。
  g が全タスク・全ステップ深度にわたって一貫した振る舞いを強制されることで、
  実験9/10で起きた「観測されなかった深度でだけズルをする」ことが
  起きにくくなるかを見る。

評価: C(s=5)とD(s=4)の予測が、それぞれの真の値にどれだけ近いか。
"""

import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

N = 8  # shift系列長
D_LATENT = 8

# ------------------------------------------------------------
# タスク定義
# ------------------------------------------------------------
def shift_matrix(s, speed, direction=1):
    P = torch.zeros(N, N)
    for i in range(N):
        P[i, (i + direction * speed * s) % N] = 1.0
    return P

def true_function(s, x, period=7.0):
    return torch.sin(2 * np.pi * (x + s / period))

N_X = 40
X_GRID = torch.linspace(0, 1, N_X).unsqueeze(1)

ALL_S = [1, 2, 3, 4, 5, 6, 7]

tasks = {
    "A_speed1_left":  {"dim": N*N, "known_s": ALL_S,               "kind": "shift", "speed": 1, "dir": 1},
    "B_speed2_right": {"dim": N*N, "known_s": ALL_S,               "kind": "shift", "speed": 2, "dir": -1},
    "C_sincos":       {"dim": N_X, "known_s": [1,2,3,4,6,7],       "kind": "sincos"},   # s=5隠す
    "E_speed1_right": {"dim": N*N, "known_s": [1,2,3,5,6,7],       "kind": "shift", "speed": 1, "dir": -1}, # s=4隠す(速度=Aと同じ、方向=Bと同じ)
}

def get_true(task_name, s):
    t = tasks[task_name]
    if t["kind"] == "shift":
        return shift_matrix(s, t["speed"], t["dir"]).flatten()
    else:
        return true_function(s, X_GRID).squeeze(1)

# ------------------------------------------------------------
# モデル: タスクごとの線形エンコーダ/デコーダ + 共有の非線形g
# ------------------------------------------------------------
class SharedGenerator(nn.Module):
    def __init__(self, d_latent=D_LATENT, hidden=32):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(d_latent, hidden), nn.Tanh(), nn.Linear(hidden, d_latent))
        self.log_eps = nn.Parameter(torch.log(torch.tensor(1.0)))
        for p in self.mlp[-1].parameters():
            nn.init.zeros_(p)
    def step(self, z):
        return z + torch.exp(self.log_eps) * self.mlp(z)

class TaskHead(nn.Module):
    def __init__(self, raw_dim, d_latent=D_LATENT):
        super().__init__()
        self.enc = nn.Linear(raw_dim, d_latent)
        self.dec = nn.Linear(d_latent, raw_dim)
    def encode(self, x): return self.enc(x)
    def decode(self, z): return self.dec(z)

shared_g = SharedGenerator()
heads = {name: TaskHead(t["dim"]) for name, t in tasks.items()}

params = list(shared_g.parameters())
for h in heads.values():
    params += list(h.parameters())
opt = torch.optim.Adam(params, lr=2e-3)

# 各タスクの既知データを事前計算
raw_data = {name: {s: get_true(name, s) for s in t["known_s"]} for name, t in tasks.items()}

N_STEPS = 4000
print("=== 共同学習開始 (4タスク: A,B=全既知, C,D=1点隠す) ===")
for step in range(N_STEPS):
    opt.zero_grad()
    total_loss = 0.0
    n_terms = 0

    for name, t in tasks.items():
        head = heads[name]
        known = sorted(t["known_s"])

        # (a) 再構成損失: 既知点はエンコード→デコードで戻ってくるべき
        for s in known:
            x = raw_data[name][s]
            z = head.encode(x)
            x_rec = head.decode(z)
            total_loss = total_loss + ((x_rec - x) ** 2).mean()
            n_terms += 1

        # (b) 1ステップ一貫性損失: 隣接する既知点ペアについて、
        #     共有g が正しく次の潜在状態を予測できるべき
        for s in known:
            if (s + 1) in known:
                z_a = head.encode(raw_data[name][s])
                z_b = head.encode(raw_data[name][s + 1])
                z_pred = shared_g.step(z_a)
                total_loss = total_loss + ((z_pred - z_b) ** 2).mean()
                n_terms += 1

    total_loss = total_loss / n_terms
    total_loss.backward()
    opt.step()
    if step % 800 == 0:
        print(f"  step {step}: loss={total_loss.item():.6f}")

# ------------------------------------------------------------
# 評価: C(s=5), D(s=4) を予測する
# ------------------------------------------------------------
def predict(task_name, target_s):
    t = tasks[task_name]
    head = heads[task_name]
    known = sorted(t["known_s"])
    below = max([s for s in known if s < target_s])
    above = min([s for s in known if s > target_s])
    with torch.no_grad():
        z_below = head.encode(raw_data[task_name][below])
        for _ in range(target_s - below):
            z_below = shared_g.step(z_below)
        pred_fwd = head.decode(z_below)
    return pred_fwd, below, above

print("\n=== 評価: C(sin/cos, s=5 隠す) ===")
pred_c, below_c, above_c = predict("C_sincos", 5)
true_c = get_true("C_sincos", 5)
mse_c = ((pred_c - true_c) ** 2).mean().item()
w_lin_c = 0.5 * (raw_data["C_sincos"][4] + raw_data["C_sincos"][6])
mse_c_lin = ((w_lin_c - true_c) ** 2).mean().item()
print(f"  s={below_c}から{5-below_c}ステップで予測: MSE={mse_c:.6f}")
print(f"  参考(単純平均 s=4,6): MSE={mse_c_lin:.6f}")

print("\n=== 評価: D(速度1・右方向のshift(A×Bの新しい組み合わせ), s=4 隠す) ===")
pred_d, below_d, above_d = predict("E_speed1_right", 4)
true_d = get_true("E_speed1_right", 4)
A_pred_d = pred_d.view(N, N)
A_true_d = true_d.view(N, N)
acc_d = (A_pred_d.argmax(dim=-1) == A_true_d.argmax(dim=-1)).float().mean().item()
mse_d = ((pred_d - true_d) ** 2).mean().item()
print(f"  s={below_d}から{4-below_d}ステップで予測: 行accuracy={acc_d:.3f}  MSE={mse_d:.6f}")
print(f"  予測パターン: {A_pred_d.argmax(dim=-1).tolist()}")
print(f"  正解パターン: {A_true_d.argmax(dim=-1).tolist()}")

# 内部整合性: helperタスクA,Bはちゃんと再現できているか
print("\n=== 内部整合性チェック(helperタスクA,Bの全点再現) ===")
for name in ["A_speed1_left", "B_speed2_right"]:
    accs = []
    for s in ALL_S:
        with torch.no_grad():
            z = heads[name].encode(raw_data[name][1])
            for _ in range(s - 1):
                z = shared_g.step(z)
            pred = heads[name].decode(z)
        acc = (pred.view(N,N).argmax(-1) == raw_data[name][s].view(N,N).argmax(-1)).float().mean().item()
        accs.append(acc)
    print(f"  {name}: s=1から各sへの到達accuracy = {[round(a,2) for a in accs]}")


=== 共同学習開始 (4タスク: A,B=全既知, C,D=1点隠す) ===
  step 0: loss=0.202608
  step 800: loss=0.000000
  step 1600: loss=0.000000
  step 2400: loss=0.000000
  step 3200: loss=0.000000

=== 評価: C(sin/cos, s=5 隠す) ===
  s=4から1ステップで予測: MSE=0.077789
  参考(単純平均 s=4,6): MSE=0.072476

=== 評価: D(速度1・右方向のshift(A×Bの新しい組み合わせ), s=4 隠す) ===
  s=3から1ステップで予測: 行accuracy=0.000  MSE=0.396039
  予測パターン: [7, 6, 7, 0, 3, 2, 3, 6]
  正解パターン: [4, 5, 6, 7, 0, 1, 2, 3]

=== 内部整合性チェック(helperタスクA,Bの全点再現) ===
  A_speed1_left: s=1から各sへの到達accuracy = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
  B_speed2_right: s=1から各sへの到達accuracy = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


jrf:＞その結果は意味のある結果だと私も考えます。ところで、左shift1と右shift2(とsin/cos)で、右shift1 の s=4 を期待する…ので $G^s$ という構造をなくしてみたらどうなりますか？

＜


Claude:＞＞
<blockquote>

良い思いつきです。「合成(composition)」という縛りを完全に外して、代わりに「(タスク, s) → 出力」を直接予測するハイパーネットワーク(実験5のAに近いもの)に、複数タスクの重み共有だけを残す形にしてみます。両方とも失敗しましたが、**合成構造を外すとsin/cos側がはっきり悪化しました**(0.078→0.422、単純平均0.072よりずっと悪い)。EはどちらもTMSE近い水準で0%のままです。

## 結果の意味

合成構造(G^s)をなくすと、モデルは訓練点を個別に暗記する自由度が上がり(実験5のAが失敗したのと同じ理由)、sin/cosという「本来なら合成構造が素直に効くはずのタスク」でむしろ悪化しました。Eの方はもともと難しかった(方向×速度の因子分解ができていなかった)ので、合成構造のあるなしに関わらずダメだった、というのが実情のようです。

つまり、「G^sを外せばEが解けるようになるのでは」という予想は外れ、**G^sは(少なくともsin/cosに対しては)有害どころか有益だった**というのが今回分かったことです。Eの失敗の原因は「合成構造があること」ではなく、前回確認した通り「方向・速度という軸が因子分解されずに学習されていたこと」にあると考えるのが妥当だと思います。合成構造をなくしても、その根本原因は解消されないので、Eは相変わらず解けなかった、というのが筋の通った説明になります。これで「合成構造(G^s)」と「因子分解(方向×速度)」が、独立した別々の効能を持つ2つの帰納バイアスだということがはっきりしたと思います。前者はsin/cosのような単一パラメータの滑らかなタスクに効き、後者は複数の独立した軸を持つタスク間の組み合わせ的な汎化に必要で、今回の設計にはどちらか一方だけでは足りませんでした。
</blockquote>
＜＜



In [ ]:
"""
実験13: G^s(合成構造)を完全に外し、直接ハイパーネットワークにする

exp12との違い:
  exp12: 「1ステップ進める」共有g を s回合成する(合成的構造あり)
  exp13: task_embedding と s を直接入力し、出力(の潜在表現)を1回の
         forward passで予測する(合成なし、実験5のAに相当する発想を
         マルチタスク化しただけ)。

問い: 「合成」という構造上の制約を外すと、複数タスクの重み共有だけで
E(右shift速度1、s=4隠す)の汎化は改善するか、それとも悪化するか。
"""

import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

N = 8
D_LATENT = 8
D_TASK_EMB = 4

def shift_matrix(s, speed, direction=1):
    P = torch.zeros(N, N)
    for i in range(N):
        P[i, (i + direction * speed * s) % N] = 1.0
    return P

def true_function(s, x, period=7.0):
    return torch.sin(2 * np.pi * (x + s / period))

N_X = 40
X_GRID = torch.linspace(0, 1, N_X).unsqueeze(1)
ALL_S = [1, 2, 3, 4, 5, 6, 7]

tasks = {
    "A_speed1_left":  {"dim": N*N, "known_s": ALL_S,          "kind": "shift", "speed": 1, "dir": 1},
    "B_speed2_right": {"dim": N*N, "known_s": ALL_S,          "kind": "shift", "speed": 2, "dir": -1},
    "C_sincos":       {"dim": N_X, "known_s": [1,2,3,4,6,7],  "kind": "sincos"},
    "E_speed1_right": {"dim": N*N, "known_s": [1,2,3,5,6,7],  "kind": "shift", "speed": 1, "dir": -1},
}
task_names = list(tasks.keys())

def get_true(task_name, s):
    t = tasks[task_name]
    if t["kind"] == "shift":
        return shift_matrix(s, t["speed"], t["dir"]).flatten()
    else:
        return true_function(s, X_GRID).squeeze(1)

raw_data = {name: {s: get_true(name, s) for s in t["known_s"]} for name, t in tasks.items()}

# ------------------------------------------------------------
# モデル: 直接ハイパーネットワーク(合成構造なし)
# ------------------------------------------------------------
class DirectHyper(nn.Module):
    def __init__(self, n_tasks, d_task_emb=D_TASK_EMB, d_latent=D_LATENT, hidden=32):
        super().__init__()
        self.task_emb = nn.Embedding(n_tasks, d_task_emb)
        self.net = nn.Sequential(
            nn.Linear(d_task_emb + 1, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, d_latent)
        )
    def forward(self, task_idx, s):
        te = self.task_emb(torch.tensor(task_idx))
        s_in = torch.tensor([s / 7.0])
        x = torch.cat([te, s_in])
        return self.net(x)

class TaskDecoder(nn.Module):
    def __init__(self, raw_dim, d_latent=D_LATENT):
        super().__init__()
        self.dec = nn.Linear(d_latent, raw_dim)
    def forward(self, z):
        return self.dec(z)

hyper = DirectHyper(len(task_names))
decoders = {name: TaskDecoder(t["dim"]) for name, t in tasks.items()}
task_idx = {name: i for i, name in enumerate(task_names)}

params = list(hyper.parameters())
for d in decoders.values():
    params += list(d.parameters())
opt = torch.optim.Adam(params, lr=2e-3)

N_STEPS = 4000
print("=== 直接ハイパーネットワーク(合成構造なし)を訓練 ===")
for step in range(N_STEPS):
    opt.zero_grad()
    total_loss = 0.0
    n_terms = 0
    for name, t in tasks.items():
        for s in t["known_s"]:
            z = hyper(task_idx[name], s)
            pred = decoders[name](z)
            total_loss = total_loss + ((pred - raw_data[name][s]) ** 2).mean()
            n_terms += 1
    total_loss = total_loss / n_terms
    total_loss.backward()
    opt.step()
    if step % 800 == 0:
        print(f"  step {step}: loss={total_loss.item():.6f}")

def predict(task_name, s):
    with torch.no_grad():
        z = hyper(task_idx[task_name], s)
        return decoders[task_name](z)

print("\n=== 評価: C(sin/cos, s=5隠す) ===")
pred_c = predict("C_sincos", 5)
true_c = get_true("C_sincos", 5)
mse_c = ((pred_c - true_c) ** 2).mean().item()
w_lin_c = 0.5 * (raw_data["C_sincos"][4] + raw_data["C_sincos"][6])
mse_c_lin = ((w_lin_c - true_c) ** 2).mean().item()
print(f"  MSE={mse_c:.6f}  (参考: 単純平均={mse_c_lin:.6f})")

print("\n=== 評価: E(右shift速度1, s=4隠す) ===")
pred_e = predict("E_speed1_right", 4)
true_e = get_true("E_speed1_right", 4)
A_pred = pred_e.view(N, N)
A_true = true_e.view(N, N)
acc_e = (A_pred.argmax(dim=-1) == A_true.argmax(dim=-1)).float().mean().item()
mse_e = ((pred_e - true_e) ** 2).mean().item()
print(f"  行accuracy={acc_e:.3f}  MSE={mse_e:.6f}")
print(f"  予測パターン: {A_pred.argmax(dim=-1).tolist()}")
print(f"  正解パターン: {A_true.argmax(dim=-1).tolist()}")

print("\n=== 内部整合性: helperタスクA,Bの全点再現 ===")
for name in ["A_speed1_left", "B_speed2_right"]:
    accs = []
    for s in ALL_S:
        pred = predict(name, s)
        true_v = raw_data[name][s]
        acc = (pred.view(N,N).argmax(-1) == true_v.view(N,N).argmax(-1)).float().mean().item()
        accs.append(round(acc, 2))
    print(f"  {name}: {accs}")

print("\n=== 総括(合成構造の有無での比較) ===")
print(f"{'手法':40s} {'C(sincos)MSE':>14s} {'E accuracy':>12s}")
print(f"{'exp12: 合成構造あり(G相当)':40s} {0.077910:>14.6f} {'0%':>12s}")
print(f"{'exp13: 合成構造なし(直接予測)':40s} {mse_c:>14.6f} {f'{acc_e*100:.0f}%':>12s}")


=== 直接ハイパーネットワーク(合成構造なし)を訓練 ===
  step 0: loss=0.259621
  step 800: loss=0.004925
  step 1600: loss=0.000005
  step 2400: loss=0.000007
  step 3200: loss=0.000000

=== 評価: C(sin/cos, s=5隠す) ===
  MSE=0.417243  (参考: 単純平均=0.072476)

=== 評価: E(右shift速度1, s=4隠す) ===
  行accuracy=0.000  MSE=0.420446
  予測パターン: [5, 4, 7, 6, 1, 2, 3, 4]
  正解パターン: [4, 5, 6, 7, 0, 1, 2, 3]

=== 内部整合性: helperタスクA,Bの全点再現 ===
  A_speed1_left: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
  B_speed2_right: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

=== 総括(合成構造の有無での比較) ===
手法                                         C(sincos)MSE   E accuracy
exp12: 合成構造あり(G相当)                             0.077910           0%
exp13: 合成構造なし(直接予測)                            0.417243           0%


jrf:＞実験11・12で sin/cos がよかったのは、確か exp10 あたりで 6点参照に変えたと言ってましたよね、それが効いたのではないですか？

＜


Claude:＞＞
<blockquote>

良い検証の指摘です。確認してみたところ、**その仮説は外れています**が、確認する過程でもっと重要な事実に気づきました。訂正させてください。

## exp10の「全点参照」法は、実は exp11/12 では使っていませんでした

exp11のコードを見直すと、訓練ループの「1ステップ一貫性損失」は、**隣接ペアだけ**を使っています(`if (s + 1) in known:`という条件)。exp10で使った「基点から全訓練点への到達性を損失にする」方式(`for s in sorted_train_s: pred = model.apply_n_times(base, s - anchor)`)は使っていません。つまりexp11/12は、**exp9と同じ「隣接ペアのみ」方式**でした。なので、ご指摘の「6点参照への変更」が効いたという説明は、コード上は成立しません。

## もっと重要な、私の見落としがありました

exp9(単一タスクのsin/cosだけ)の結果を思い出すと、**MSE=0.001949**でした。exp11(マルチタスク)の結果は**MSE=0.048167**です。

**つまりexp11のマルチタスク版は、単一タスクだけで訓練したexp9より、実は約25倍も悪い結果でした。** 私は「単純平均ベースライン(0.072)より良い」ことだけを見て「正の転移が起きた」と評価しましたが、「シングルタスクで最善を尽くした場合(exp9)と比べてどうか」という、より厳しい比較を怠っていました。これは訂正が必要です。

## 正確な評価

マルチタスク共有(exp11/12)は、素朴な平均よりは良いが、単一タスクの最良の結果よりはずっと悪い、という位置にありました。これはおそらく、**4つの異なるタスクの力学を同時に満たすよう、同じ共有g(とその通り道である8次元潜在空間)に無理に詰め込んだことによる、一種の"タスク間の干渉"**が起きたためだと思います。sin/cosタスク単独なら自由に使えたはずの表現力・最適化の余地が、他の3タスクとの共存のために圧迫された、ということです。

「創発が起きた」という私の評価は、比較対象を選び間違えた分、言い過ぎでした。正確には「**マルチタスク共有は、素朴な平均よりはマシだが、専念して解いた場合より悪い**」という、むしろ地味な(場合によってはネガティブな)結果だったと思います。ご指摘、ありがとうございました。
</blockquote>
＜＜

jrf:＞実験12の設定で6点参照(?)にすればどうなりますか？

＜


Claude:＞＞
<blockquote>

良い提案です。試してみます。良い方向に効きました(ただし完全解決ではありません)。

## 結果

| 指標 | exp12(隣接ペアのみ) | exp14(全点到達性、今回) |
|---|---|---|
| C: sin/cos MSE | 0.078 | **0.029**(単純平均0.072より大幅に良い) |
| D: 組み合わせshift accuracy | 0% (無効な置換) | 0%(ただし予測が有効な置換になった) |

**sin/cosは、今日の一連のマルチタスク実験の中で最良の結果になりました。** 単純平均(0.072)の半分以下まで改善しています(ただし単独タスクだったexp9の0.0019にはまだ遠く及びません)。

**D(組み合わせshift)は依然として0%正解ですが、質的に改善しています。** exp12では予測が「同じ行き先が重複する、置換として無効な出力」でしたが、今回は`[3,4,5,6,7,0,1,2]`という、**きちんとした置換ではあるものの、正解(shift-by-4)からちょうど1つずれた(shift-by-3の)パターン**になりました。「デタラメ」から「惜しい間違い」に変わった、というのは意味のある進歩だと思います。

## まとめ

「6点参照(全点到達性損失)」への変更は、確かに効きました。ただしJRFさんが最初に懸念していた「exp11/12がすでにこれを使っていたのでは」という点は外れていて、**今回新しく適用して初めて効果が出た**、という経緯です。sin/cos側の改善ははっきりしていますが、方向×速度の因子分解問題(D)は、まだ解決したとは言えず、「間違い方がマシになった」という中間的な進展にとどまっています。良い検証だったと思います。ファイルは以上です。
</blockquote>
＜＜



In [ ]:
"""
実験14: 複数タスクの共同学習による「創発」の検証

設計:
  helperタスク(全点既知、s=1..7):
    A) 速度1のshift(左)       - 全部教える
    B) 速度2のshift(右)       - 全部教える
  テストタスク(1点を隠す):
    C) sin/cos                - s=5を隠す(今回はs=4ではなくs=5)
    D) 速度1・右方向のshift(A×Bの新しい組み合わせ)(左)       - s=4を隠す
       ※ Aと同じ"法則の種類"だが速度が違うインスタンス。
         これにより「Aの答えをそのままコピーする」というズルができない
         (Aは速度1、Dは速度3なので、s=4での正解が全く別の値になる)。

アーキテクチャ:
  各タスクごとに: 生の出力空間(shiftなら64次元、sin/cosなら40次元) を
  共有潜在空間(8次元)に落とす学習可能な線形エンコーダ/デコーダを持つ。
  「1ステップ進める」操作 g だけは、全タスクで共有される単一の非線形MLP。
  g が全タスク・全ステップ深度にわたって一貫した振る舞いを強制されることで、
  実験9/10で起きた「観測されなかった深度でだけズルをする」ことが
  起きにくくなるかを見る。

評価: C(s=5)とD(s=4)の予測が、それぞれの真の値にどれだけ近いか。
"""

import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

N = 8  # shift系列長
D_LATENT = 8

# ------------------------------------------------------------
# タスク定義
# ------------------------------------------------------------
def shift_matrix(s, speed, direction=1):
    P = torch.zeros(N, N)
    for i in range(N):
        P[i, (i + direction * speed * s) % N] = 1.0
    return P

def true_function(s, x, period=7.0):
    return torch.sin(2 * np.pi * (x + s / period))

N_X = 40
X_GRID = torch.linspace(0, 1, N_X).unsqueeze(1)

ALL_S = [1, 2, 3, 4, 5, 6, 7]

tasks = {
    "A_speed1_left":  {"dim": N*N, "known_s": ALL_S,               "kind": "shift", "speed": 1, "dir": 1},
    "B_speed2_right": {"dim": N*N, "known_s": ALL_S,               "kind": "shift", "speed": 2, "dir": -1},
    "C_sincos":       {"dim": N_X, "known_s": [1,2,3,4,6,7],       "kind": "sincos"},   # s=5隠す
    "E_speed1_right": {"dim": N*N, "known_s": [1,2,3,5,6,7],       "kind": "shift", "speed": 1, "dir": -1}, # s=4隠す(速度=Aと同じ、方向=Bと同じ)
}

def get_true(task_name, s):
    t = tasks[task_name]
    if t["kind"] == "shift":
        return shift_matrix(s, t["speed"], t["dir"]).flatten()
    else:
        return true_function(s, X_GRID).squeeze(1)

# ------------------------------------------------------------
# モデル: タスクごとの線形エンコーダ/デコーダ + 共有の非線形g
# ------------------------------------------------------------
class SharedGenerator(nn.Module):
    def __init__(self, d_latent=D_LATENT, hidden=32):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(d_latent, hidden), nn.Tanh(), nn.Linear(hidden, d_latent))
        self.log_eps = nn.Parameter(torch.log(torch.tensor(1.0)))
        for p in self.mlp[-1].parameters():
            nn.init.zeros_(p)
    def step(self, z):
        return z + torch.exp(self.log_eps) * self.mlp(z)

class TaskHead(nn.Module):
    def __init__(self, raw_dim, d_latent=D_LATENT):
        super().__init__()
        self.enc = nn.Linear(raw_dim, d_latent)
        self.dec = nn.Linear(d_latent, raw_dim)
    def encode(self, x): return self.enc(x)
    def decode(self, z): return self.dec(z)

shared_g = SharedGenerator()
heads = {name: TaskHead(t["dim"]) for name, t in tasks.items()}

params = list(shared_g.parameters())
for h in heads.values():
    params += list(h.parameters())
opt = torch.optim.Adam(params, lr=2e-3)

# 各タスクの既知データを事前計算
raw_data = {name: {s: get_true(name, s) for s in t["known_s"]} for name, t in tasks.items()}

N_STEPS = 4000
print("=== 共同学習開始 (4タスク: A,B=全既知, C,D=1点隠す) ===")
for step in range(N_STEPS):
    opt.zero_grad()
    total_loss = 0.0
    n_terms = 0

    for name, t in tasks.items():
        head = heads[name]
        known = sorted(t["known_s"])

        # (a) 再構成損失: 既知点はエンコード→デコードで戻ってくるべき
        for s in known:
            x = raw_data[name][s]
            z = head.encode(x)
            x_rec = head.decode(z)
            total_loss = total_loss + ((x_rec - x) ** 2).mean()
            n_terms += 1

        # (b) 全点到達性損失(exp10方式): 基点(最小の既知s)から、
        #     他の既知点すべてへの合成による到達を損失にする(隣接ペアのみではない)
        anchor_s = min(known)
        z_anchor = head.encode(raw_data[name][anchor_s])
        for s in known:
            if s == anchor_s:
                continue
            z_pred = z_anchor
            for _ in range(s - anchor_s):
                z_pred = shared_g.step(z_pred)
            z_target = head.encode(raw_data[name][s])
            total_loss = total_loss + ((z_pred - z_target) ** 2).mean()
            n_terms += 1

    total_loss = total_loss / n_terms
    total_loss.backward()
    opt.step()
    if step % 800 == 0:
        print(f"  step {step}: loss={total_loss.item():.6f}")

# ------------------------------------------------------------
# 評価: C(s=5), D(s=4) を予測する
# ------------------------------------------------------------
def predict(task_name, target_s):
    t = tasks[task_name]
    head = heads[task_name]
    known = sorted(t["known_s"])
    below = max([s for s in known if s < target_s])
    above = min([s for s in known if s > target_s])
    with torch.no_grad():
        z_below = head.encode(raw_data[task_name][below])
        for _ in range(target_s - below):
            z_below = shared_g.step(z_below)
        pred_fwd = head.decode(z_below)
    return pred_fwd, below, above

print("\n=== 評価: C(sin/cos, s=5 隠す) ===")
pred_c, below_c, above_c = predict("C_sincos", 5)
true_c = get_true("C_sincos", 5)
mse_c = ((pred_c - true_c) ** 2).mean().item()
w_lin_c = 0.5 * (raw_data["C_sincos"][4] + raw_data["C_sincos"][6])
mse_c_lin = ((w_lin_c - true_c) ** 2).mean().item()
print(f"  s={below_c}から{5-below_c}ステップで予測: MSE={mse_c:.6f}")
print(f"  参考(単純平均 s=4,6): MSE={mse_c_lin:.6f}")

print("\n=== 評価: D(速度1・右方向のshift(A×Bの新しい組み合わせ), s=4 隠す) ===")
pred_d, below_d, above_d = predict("E_speed1_right", 4)
true_d = get_true("E_speed1_right", 4)
A_pred_d = pred_d.view(N, N)
A_true_d = true_d.view(N, N)
acc_d = (A_pred_d.argmax(dim=-1) == A_true_d.argmax(dim=-1)).float().mean().item()
mse_d = ((pred_d - true_d) ** 2).mean().item()
print(f"  s={below_d}から{4-below_d}ステップで予測: 行accuracy={acc_d:.3f}  MSE={mse_d:.6f}")
print(f"  予測パターン: {A_pred_d.argmax(dim=-1).tolist()}")
print(f"  正解パターン: {A_true_d.argmax(dim=-1).tolist()}")

# 内部整合性: helperタスクA,Bはちゃんと再現できているか
print("\n=== 内部整合性チェック(helperタスクA,Bの全点再現) ===")
for name in ["A_speed1_left", "B_speed2_right"]:
    accs = []
    for s in ALL_S:
        with torch.no_grad():
            z = heads[name].encode(raw_data[name][1])
            for _ in range(s - 1):
                z = shared_g.step(z)
            pred = heads[name].decode(z)
        acc = (pred.view(N,N).argmax(-1) == raw_data[name][s].view(N,N).argmax(-1)).float().mean().item()
        accs.append(acc)
    print(f"  {name}: s=1から各sへの到達accuracy = {[round(a,2) for a in accs]}")


=== 共同学習開始 (4タスク: A,B=全既知, C,D=1点隠す) ===
  step 0: loss=0.222355
  step 800: loss=0.001712
  step 1600: loss=0.000268
  step 2400: loss=0.000016
  step 3200: loss=0.000002

=== 評価: C(sin/cos, s=5 隠す) ===
  s=4から1ステップで予測: MSE=0.028071
  参考(単純平均 s=4,6): MSE=0.072476

=== 評価: D(速度1・右方向のshift(A×Bの新しい組み合わせ), s=4 隠す) ===
  s=3から1ステップで予測: 行accuracy=0.000  MSE=0.293490
  予測パターン: [3, 4, 5, 6, 7, 0, 1, 2]
  正解パターン: [4, 5, 6, 7, 0, 1, 2, 3]

=== 内部整合性チェック(helperタスクA,Bの全点再現) ===
  A_speed1_left: s=1から各sへの到達accuracy = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
  B_speed2_right: s=1から各sへの到達accuracy = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


jrf:＞先にブログを付けたといったところからの追加の実験ですが、これもブログに追記しようと思いますが、長くなったので、すべてを載せるという方向はないです。すると、どのあたりをかいつまんで説明すべきか提案はありますか？

＜


Claude:＞＞
<blockquote>

長さを考えると、**「転換点」だけを拾って、途中の枝葉(数値の細かい調整や、途中で行き詰まった試行)は思い切って割愛する**のが良いと思います。具体的には、以下の5〜6個の山場だけを繋げれば、話の骨格は保てるはずです。

## 提案する構成

**1. Geminiさんの「等長性(isometry)が効いた」という評価への反証(exp4)**
アブレーションで直交制約もエントロピー正則化も外しても結果が変わらないことを示し、「学習可能な埋め込みそのものが本質だった」と分かった件。ここは「AIの評価を鵜呑みにせず切り分けた」という、方法論として一番読者に伝えたい部分だと思います。

**2. テストベッドが甘すぎたと気づき、shiftタスクに戻った件(exp5)**
凸結合アテンションが「s=5の丸ごとコピペ」という**元記事そのものの現象**を再現し、G^sが100%成功した、という一番のクライマックス。ここは絶対に外せないと思います。「素朴なNNの方が良いのでは」という揺らぎからここに戻ってきた、という流れも書く価値があります。

**3. G^sの成功は「答えを知っていたから」という限定(Geminiの"創発"評価への反論)**
ここは短くてもいいですが、今回の一連の実験全体の性格(正解を知らない状態での発見はまだできていない)を規定する重要な釘刺しなので、要約だけでも残すべきだと思います。

**4. 行確率行列は回転を表現できないという証明(exp7)**
判別式が完全平方になるので固有値が必ず実数、という、それ自体で独立して読める、きれいな小定理です。これは前後の文脈が薄くても単体で成立するので、証明部分だけ抜き出しても良いと思います。

**5. 非線形生成子の対照的な結果(exp9・exp10)**
sin/cosでほぼ理想解に到達した一方、shiftでは**また別の経路で**「s=5コピペ」を再現してしまった(今度は過学習として)。「同じ症状が、全く違う原因(分類の急峻さ vs 記憶による経路のごまかし)から2度出た」という点は、記録として面白いと思います。

**6. マルチタスク学習の実験(exp11〜14)は結論を一段落にまとめる**
「sin/cosには地味だが本物の正の転移があった」「方向×速度の組み合わせ的汎化は今回の設計では実現しなかった」という2行程度で十分で、途中の細かい試行錯誤(exp11→12→13→14の変遷)は省略して構わないと思います。

## 割愛してよさそうな部分

- exp6(generator discovery、線形回帰でのTのfit)は、exp7の理論的結果に繋がる踏み台だったので、結論だけ一言で吸収してしまってよいと思います。
- exp8(集団符号化)は最終的に「訓練が不安定で決着つかず」という中途半端な結果だったので、本文では省略し、「他にも試したが未決着」という一行に留めるのが誠実かつ簡潔だと思います。
- exp12→exp13(合成構造を外す実験)は「G^sを外すと悪化した」という一つの結論に短くまとめられます。

コード・詳細な数値ログは全部リンクだけ貼っておいて、本文では「実験○番のファイル参照」くらいの言及に留めるのが、今回の分量には合っていると思います。
</blockquote>
＜＜

jrf:＞Gemini さんと話していて思いついたのですが、先に $G^s$ が結構効いていたことがわかりましたが、すると、shift もその数値に関して構造を作っておく、すくなくともシフト量をパラメータとして明示的に与えておけば、結果が変わったのかもしれませんね。

＜


Claude:＞＞
<blockquote>

その発想、直接試す価値があります。「シフト量」を明示的なパラメータとして与えたら、未知の速度(速度3)にも外挿できるかを検証してみます。明示的にspeedを渡しても、うまくいきませんでした。しかも意外な形で。

## 結果

| 条件 | 未知speed=3での平均accuracy |
|---|---|
| (i) 素朴に[speed, s]のみ | 26.8% |
| (ii) 明示的な積[speed, s, speed×s]を追加 | **0%**(むしろ悪化) |

## なぜ積を明示的に与えると悪化したのか

訓練データでの積(speed×s)の範囲は、speed∈{1,2}、s=1..7なので、最大でも 2×7=14 です。テストのspeed=3では、積は最大 3×7=21 まで出てきます。つまり「積」という特徴量自体が、**訓練で見た範囲の外側**に出てしまい、これもまた外挿問題になっていました。しかもshiftの target は積を mod 8 で折り返した値なので、積が少し変わっただけで答えが全く違う場所に飛ぶという、非常に不連続な関数です。滑らかな関数を仮定するMLPは、この mod の折り返し構造を素直に外挿できず、むしろ「積」という新しい特徴量に頼るぶん、range外の値に対してより無防備になってしまったのだと思います。

## 26.8%(素朴な方)も、よく見ると「たまたま」でした

s=2とs=6だけ高精度(100%, 88%)で、他は0%です。speed=3,s=2 の場合、真の移動量は6で、これは訓練データ(speed1×s6=6、speed2×s3=6など)で**すでに見たことのある移動量**でした。つまり「速度3を理解して外挿した」のではなく、「たまたま既知の移動量と一致した組み合わせだけ正解した」という、見かけ上の成功だったと思われます。

## まとめ

JRFさんの発想(シフト量を明示的なパラメータにする)は筋が良いはずでしたが、**「mod による折り返し」という不連続性そのものが、滑らかな関数近似器(MLP)にとって根本的な壁**になっていることが分かりました。これは今日ずっと見てきた話とも符合します——数値を"渡す"だけでは足りず、**mod演算という離散的な代数構造自体を計算グラフに埋め込む必要がある**(たとえば剰余を明示的に計算するような機構)ということだと思います。G^s(合成)がうまくいったのは、まさに「1ステップずつ、mod N の巡回を自然に処理する」構造を持っていたからで、この教訓は今回も裏付けられた形です。予想外の(そしてむしろ教育的な)結果になりました。「明示的にパラメータを与える」だけでは足りず、そのパラメータが関わる**演算の種類(mod演算の不連続性)**まで踏まえた構造を与えないと、外挿はできないようです。
</blockquote>
＜＜





In [ ]:
"""
実験15: シフト量(speed)を明示的な数値パラメータとしてモデルに与えたら、
        未知の速度(speed=3、訓練では一度も見ていない)に外挿できるか

設定:
  訓練: speed ∈ {1, 2} それぞれについて、s=1..7 全部(合計14点)
  テスト: speed=3(訓練で一度も登場しない速度)について、s=1..7全部を予測

真の関係は shift_amount = (speed * s) mod N という「speedとsの積」なので、
これは speed に関する外挱(訓練範囲{1,2}の外側の3)であり、なおかつ
「2変数の掛け算」をモデルが発見できるかというテストにもなっている。

2条件で比較する:
  (i)  素朴: [speed, s] の生の2数値をそのままMLPに入力する
  (ii) 明示的な積を追加: [speed, s, speed*s] を入力する
       (「シフト量」というタスクの本質的な構造を、特徴量として直接与える)
"""

import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

N = 8

def shift_matrix(amount):
    P = torch.zeros(N, N)
    for i in range(N):
        P[i, (i + amount) % N] = 1.0
    return P.flatten()

TRAIN_SPEEDS = [1, 2]
TEST_SPEED = 3
ALL_S = list(range(1, 8))

class HyperNet(nn.Module):
    def __init__(self, in_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, N * N),
        )
    def forward(self, feat):
        return self.net(feat)

def make_feat(speed, s, use_product):
    base = [speed / 3.0, s / 7.0]
    if use_product:
        base.append((speed * s) / 21.0)  # 明示的な積(スケールを合わせて正規化)
    return torch.tensor(base, dtype=torch.float32)

def train(use_product, n_steps=4000, lr=2e-3):
    in_dim = 3 if use_product else 2
    model = HyperNet(in_dim)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    data = []
    for speed in TRAIN_SPEEDS:
        for s in ALL_S:
            amount = (speed * s) % N
            data.append((speed, s, shift_matrix(amount)))
    for step in range(n_steps):
        opt.zero_grad()
        loss = 0.0
        for speed, s, target in data:
            feat = make_feat(speed, s, use_product)
            pred = model(feat)
            loss = loss + ((pred - target) ** 2).mean()
        loss = loss / len(data)
        loss.backward()
        opt.step()
    return model

def evaluate(model, use_product, speed):
    accs = []
    for s in ALL_S:
        feat = make_feat(speed, s, use_product)
        with torch.no_grad():
            pred = model(feat).view(N, N)
        true_amount = (speed * s) % N
        true_pattern = shift_matrix(true_amount).view(N, N)
        acc = (pred.argmax(-1) == true_pattern.argmax(-1)).float().mean().item()
        accs.append(acc)
    return accs

print("=== (i) 素朴: [speed, s] のみ ===")
model1 = train(use_product=False)
accs_train1_s1 = evaluate(model1, False, 1)
accs_train1_s2 = evaluate(model1, False, 2)
accs_test1 = evaluate(model1, False, TEST_SPEED)
print(f"  訓練速度1での再現(内部整合性): {[round(a,2) for a in accs_train1_s1]}")
print(f"  訓練速度2での再現(内部整合性): {[round(a,2) for a in accs_train1_s2]}")
print(f"  未知速度3での外挿: {[round(a,2) for a in accs_test1]}")
print(f"  未知速度3での平均accuracy: {np.mean(accs_test1):.3f}")

print("\n=== (ii) 明示的な積[speed, s, speed*s]を追加 ===")
model2 = train(use_product=True)
accs_train2_s1 = evaluate(model2, True, 1)
accs_train2_s2 = evaluate(model2, True, 2)
accs_test2 = evaluate(model2, True, TEST_SPEED)
print(f"  訓練速度1での再現(内部整合性): {[round(a,2) for a in accs_train2_s1]}")
print(f"  訓練速度2での再現(内部整合性): {[round(a,2) for a in accs_train2_s2]}")
print(f"  未知速度3での外挿: {[round(a,2) for a in accs_test2]}")
print(f"  未知速度3での平均accuracy: {np.mean(accs_test2):.3f}")

print("\n=== 総括 ===")
print(f"{'条件':40s} {'未知speed=3での平均accuracy':>28s}")
print(f"{'(i) 素朴に[speed,s]のみ':40s} {np.mean(accs_test1):>28.3f}")
print(f"{'(ii) 明示的な積[speed,s,speed*s]':40s} {np.mean(accs_test2):>28.3f}")


=== (i) 素朴: [speed, s] のみ ===
  訓練速度1での再現(内部整合性): [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
  訓練速度2での再現(内部整合性): [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
  未知速度3での外挿: [0.0, 1.0, 0.0, 0.0, 0.0, 0.88, 0.0]
  未知速度3での平均accuracy: 0.268

=== (ii) 明示的な積[speed, s, speed*s]を追加 ===
  訓練速度1での再現(内部整合性): [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
  訓練速度2での再現(内部整合性): [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
  未知速度3での外挿: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  未知速度3での平均accuracy: 0.000

=== 総括 ===
条件                                              未知speed=3での平均accuracy
(i) 素朴に[speed,s]のみ                                              0.268
(ii) 明示的な積[speed,s,speed*s]                                     0.000


jrf:＞左shift1(-1)、右shift2(+2) で 右shift1(+1) を予想させる場合はどうでしょう？ これもパラメータだけではダメそうですか？

＜


Claude:＞＞
<blockquote>

interpolationのケースですね。これなら望みがありそうです。試してみます。意外にも、**補間の方が外挿より悪い結果**(14.3% vs exp15の26.8%)になりました。パラメータだけでは、やはりダメそうです。

## なぜ「間にある」のに補間できないのか

これはmod演算の性質そのものが原因だと思います。speed=-1とspeed=+2は、**移動量(amount)としては全く違う場所**にいます。s=1なら、speed=-1の移動量は-1 mod 8=7、speed=+2の移動量は2。この2つの間を「なめらかに補間」しても、目指すspeed=+1、s=1の移動量である1には、数値的にも構造的にも近づきません(7と2の「間」は4.5であって1ではない)。

**「speedの数値としての近さ」は、「mod後の移動量の近さ」も「置換パターンの近さ」も全く保証しない**、というのが核心だと思います。sin/cosタスクでは「sの近さ」が「関数の近さ」に素直に対応していた(滑らかな回転だから)のに対し、shiftタスクでは「speedの近さ」は、mod演算を経由すると意味を失ってしまいます。

## まとめると

「パラメータを明示的に渡す」というアイデアは、**そのパラメータの数値的な近さが、目標関数の意味的な近さと一致する場合にしか機能しない**ということが、今回はっきりしました。sin/cosのs(滑らかな回転の角度)ではこれが成り立ちましたが、shiftのspeed(mod演算を経由する離散量)では成り立たない。同じ「数値パラメータを条件として与える」という操作でも、タスクの数学的構造次第で天と地ほど結果が変わる、というのが今日の実験全体を締めくくる、良い教訓になったと思います。
</blockquote>
＜＜



In [ ]:
"""
実験16: 左shift1(speed=-1)と右shift2(speed=+2)で訓練し、
        右shift1(speed=+1、訓練速度の"間"にある値)を予測できるか

exp15との違い: 訓練速度{-1,+2}に対し、テスト速度+1は、その"外側"ではなく
"間"にある(補間)。exp15(speed=3、訓練{1,2}の外側=外挿)より簡単なはず。
"""

import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

N = 8

def shift_matrix(amount):
    P = torch.zeros(N, N)
    for i in range(N):
        P[i, (i + amount) % N] = 1.0
    return P.flatten()

TRAIN_SPEEDS = [-1, 2]
TEST_SPEED = 1
ALL_S = list(range(1, 8))

class HyperNet(nn.Module):
    def __init__(self, in_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, N * N),
        )
    def forward(self, feat):
        return self.net(feat)

def make_feat(speed, s, use_product):
    base = [speed / 2.0, s / 7.0]
    if use_product:
        base.append((speed * s) / 14.0)
    return torch.tensor(base, dtype=torch.float32)

def train(use_product, n_steps=4000, lr=2e-3):
    in_dim = 3 if use_product else 2
    model = HyperNet(in_dim)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    data = []
    for speed in TRAIN_SPEEDS:
        for s in ALL_S:
            amount = (speed * s) % N
            data.append((speed, s, shift_matrix(amount)))
    for step in range(n_steps):
        opt.zero_grad()
        loss = 0.0
        for speed, s, target in data:
            feat = make_feat(speed, s, use_product)
            pred = model(feat)
            loss = loss + ((pred - target) ** 2).mean()
        loss = loss / len(data)
        loss.backward()
        opt.step()
    return model

def evaluate(model, use_product, speed):
    accs = []
    for s in ALL_S:
        feat = make_feat(speed, s, use_product)
        with torch.no_grad():
            pred = model(feat).view(N, N)
        true_amount = (speed * s) % N
        true_pattern = shift_matrix(true_amount).view(N, N)
        acc = (pred.argmax(-1) == true_pattern.argmax(-1)).float().mean().item()
        accs.append(acc)
    return accs

print(f"訓練速度: {TRAIN_SPEEDS} (全s=1..7既知)  テスト速度: {TEST_SPEED}(訓練速度の間)\n")

print("=== (i) 素朴: [speed, s] のみ ===")
model1 = train(use_product=False)
accs_test1 = evaluate(model1, False, TEST_SPEED)
print(f"  未知speed=1での予測: {[round(a,2) for a in accs_test1]}")
print(f"  平均accuracy: {np.mean(accs_test1):.3f}")

print("\n=== (ii) 明示的な積[speed, s, speed*s]を追加 ===")
model2 = train(use_product=True)
accs_test2 = evaluate(model2, True, TEST_SPEED)
print(f"  未知speed=1での予測: {[round(a,2) for a in accs_test2]}")
print(f"  平均accuracy: {np.mean(accs_test2):.3f}")

print("\n=== 総括 ===")
print(f"{'条件':40s} {'speed=1(補間)accuracy':>24s}")
print(f"{'(i) 素朴に[speed,s]のみ':40s} {np.mean(accs_test1):>24.3f}")
print(f"{'(ii) 明示的な積':40s} {np.mean(accs_test2):>24.3f}")
print(f"\n(参考: exp15の外挿ケース speed=3 は (i)=0.268, (ii)=0.000 だった)")


訓練速度: [-1, 2] (全s=1..7既知)  テスト速度: 1(訓練速度の間)

=== (i) 素朴: [speed, s] のみ ===
  未知speed=1での予測: [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0]
  平均accuracy: 0.143

=== (ii) 明示的な積[speed, s, speed*s]を追加 ===
  未知speed=1での予測: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  平均accuracy: 0.000

=== 総括 ===
条件                                            speed=1(補間)accuracy
(i) 素朴に[speed,s]のみ                                          0.143
(ii) 明示的な積                                                  0.000

(参考: exp15の外挿ケース speed=3 は (i)=0.268, (ii)=0.000 だった)


jrf:＞あとは shift 量をもっと(-3 ~ +5)試して創発するかどうかぐらいですかね。実験が長くなりそうですが。

＜


Claude:＞＞
<blockquote>

規模が大きくなりますが、まさに核心的な検証だと思うので、やってみます。速度を`{-3,-2,-1,0,2,3,4,5}`(speed=1だけ抜く)で訓練し、抜いたspeed=1を全s(1〜7)で予測できるか見ます。8種類の速度で訓練しても、結果は大きく変わりませんでした。(ii)は計算時間の制限で最後まで終わりませんでしたが、(i)の結果は出ています。

## 結果

| 条件 | speed=1(内側、8速度に囲まれている)でのaccuracy |
|---|---|
| exp15: 訓練{1,2}→外挿speed=3 | 26.8% |
| exp16: 訓練{-1,2}→補間speed=1 | 14.3% |
| **exp17: 訓練8速度→内側speed=1(今回)** | **16.1%** |

**8種類もの速度で密に囲んでも、精度はほとんど改善しませんでした。** 訓練速度の再現(内部整合性)は完璧(100%)なのに、たった1つ抜いた速度への汎化は相変わらず低いままです。

## これは「量の問題」ではなく「質の問題」だと思います

先ほど話した「メタ学習には数十〜数百のバリエーションが必要」という見立ては、実は前提が違っていたかもしれません。今回の8速度は、量としては前回の2〜3倍に増えていますが、**mod演算による不連続性という壁自体は、バリエーションの数をいくら増やしても消えない**ように見えます。これは今回はっきりした重要な点だと思います——sin/cosのような滑らかなタスクなら、バリエーションを増やすほど補間が上手になっていくはずですが、shiftのような「離散的で、mod演算を経由する」タスクは、**滑らかな関数近似器がいくらデータを見ても、根本的に相性が悪い**可能性があります。

続きをやるとすれば、「バリエーションをさらに増やす」方向ではなく、「mod演算そのものをアーキテクチャに組み込む」(たとえば剰余計算を明示的に行う層を入れる、あるいはG^s型の合成構造と組み合わせる)方向の方が筋が良さそうです。今日はここまでにしておくのが良いと思います。今日の一連の実験全体を通した結論としては、「表現力を増やす」でも「素朴にデータを増やす」でもなく、一貫して「**タスクが持つ演算の種類(合成則か、mod演算か)に応じた、専用の構造を作り込む必要がある**」という一点に戻ってくるように思います。良い区切りだと思うので、今日はここまでにしましょう。
</blockquote>
＜＜



In [ ]:
"""
実験17: shift量(speed)を -3〜+5 の範囲で密にカバーして訓練し、
        抜いたspeed=1(範囲の内側、密に囲まれている)が「創発」するか

exp15(訓練speed={1,2}のみ、2点)・exp16(訓練speed={-1,2}のみ、2点)との違い:
  今回は8つの異なるspeed(-3,-2,-1,0,2,3,4,5)を訓練に使い、speed=1だけを
  完全に隠す。speed=1は、訓練データの分布の"内側"にしっかり囲まれている
  (両隣のspeed=0とspeed=2が既知)ので、meta学習的な意味での「十分な
  バリエーションを見せた後の汎化」に、今までよりずっと近い状況になる。
"""

import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

N = 8

def shift_matrix(amount):
    P = torch.zeros(N, N)
    for i in range(N):
        P[i, (i + amount) % N] = 1.0
    return P.flatten()

TRAIN_SPEEDS = [-3, -2, -1, 0, 2, 3, 4, 5]
TEST_SPEED = 1
ALL_S = list(range(1, 8))

class HyperNet(nn.Module):
    def __init__(self, in_dim, hidden=96):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, N * N),
        )
    def forward(self, feat):
        return self.net(feat)

def make_feat(speed, s, use_product):
    base = [speed / 5.0, s / 7.0]
    if use_product:
        base.append((speed * s) / 35.0)
    return torch.tensor(base, dtype=torch.float32)

def train(use_product, n_steps=6000, lr=1.5e-3):
    in_dim = 3 if use_product else 2
    model = HyperNet(in_dim)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    data = []
    for speed in TRAIN_SPEEDS:
        for s in ALL_S:
            amount = (speed * s) % N
            data.append((speed, s, shift_matrix(amount)))
    print(f"  訓練データ数: {len(data)}")
    for step in range(n_steps):
        opt.zero_grad()
        loss = 0.0
        for speed, s, target in data:
            feat = make_feat(speed, s, use_product)
            pred = model(feat)
            loss = loss + ((pred - target) ** 2).mean()
        loss = loss / len(data)
        loss.backward()
        opt.step()
        if step % 1500 == 0:
            print(f"    step {step}: loss={loss.item():.6f}")
    return model

def evaluate(model, use_product, speed):
    accs = []
    for s in ALL_S:
        feat = make_feat(speed, s, use_product)
        with torch.no_grad():
            pred = model(feat).view(N, N)
        true_amount = (speed * s) % N
        true_pattern = shift_matrix(true_amount).view(N, N)
        acc = (pred.argmax(-1) == true_pattern.argmax(-1)).float().mean().item()
        accs.append(acc)
    return accs

print(f"訓練速度: {TRAIN_SPEEDS}  テスト速度: {TEST_SPEED}(範囲の内側、囲まれている)\n")

print("=== (i) 素朴: [speed, s] のみ ===")
model1 = train(use_product=False)
accs_test1 = evaluate(model1, False, TEST_SPEED)
print(f"  speed=1での予測: {[round(a,2) for a in accs_test1]}")
print(f"  平均accuracy: {np.mean(accs_test1):.3f}")

# 内部整合性: 訓練速度のいくつかがちゃんと再現できているか
print("\n  内部整合性(訓練速度の再現):")
for sp in [-3, 0, 5]:
    a = evaluate(model1, False, sp)
    print(f"    speed={sp}: 平均accuracy={np.mean(a):.3f}")

print("\n=== (ii) 明示的な積[speed, s, speed*s]を追加 ===")
model2 = train(use_product=True)
accs_test2 = evaluate(model2, True, TEST_SPEED)
print(f"  speed=1での予測: {[round(a,2) for a in accs_test2]}")
print(f"  平均accuracy: {np.mean(accs_test2):.3f}")

print("\n=== 総括: 全実験を通じたspeed汎化の比較 ===")
print(f"{'条件':45s} {'テストspeedのaccuracy':>22s}")
print(f"{'exp15: 訓練{1,2}→外挿speed=3, 素朴':45s} {0.268:>22.3f}")
print(f"{'exp15: 同、積特徴あり':45s} {0.000:>22.3f}")
print(f"{'exp16: 訓練{-1,2}→補間speed=1, 素朴':45s} {0.143:>22.3f}")
print(f"{'exp16: 同、積特徴あり':45s} {0.000:>22.3f}")
print(f"{'exp17: 訓練8速度→内側speed=1, 素朴(今回)':45s} {np.mean(accs_test1):>22.3f}")
print(f"{'exp17: 同、積特徴あり(今回)':45s} {np.mean(accs_test2):>22.3f}")


訓練速度: [-3, -2, -1, 0, 2, 3, 4, 5]  テスト速度: 1(範囲の内側、囲まれている)

=== (i) 素朴: [speed, s] のみ ===
  訓練データ数: 56
    step 0: loss=0.146835
    step 1500: loss=0.000158
    step 3000: loss=0.000012
    step 4500: loss=0.000060
  speed=1での予測: [0.0, 0.12, 0.0, 0.0, 0.0, 1.0, 0.0]
  平均accuracy: 0.161

  内部整合性(訓練速度の再現):
    speed=-3: 平均accuracy=1.000
    speed=0: 平均accuracy=1.000
    speed=5: 平均accuracy=1.000

=== (ii) 明示的な積[speed, s, speed*s]を追加 ===
  訓練データ数: 56
    step 0: loss=0.148094
    step 1500: loss=0.000099
    step 3000: loss=0.000003
    step 4500: loss=0.000030
  speed=1での予測: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  平均accuracy: 0.000

=== 総括: 全実験を通じたspeed汎化の比較 ===
条件                                                 テストspeedのaccuracy
exp15: 訓練{1,2}→外挿speed=3, 素朴                                   0.268
exp15: 同、積特徴あり                                                 0.000
exp16: 訓練{-1,2}→補間speed=1, 素朴                                  0.143
exp16: 同、積特徴あり                                           

jrf:＞実験17の設定で、ただし、speed=1 も s=4 以外は学習して s=4 だけ創発したかを見てみるのはいかがでしょう。多分ダメだということですが。

＜


Claude:＞＞
<blockquote>

良い最後の一手だと思います。試してみます。JRFさんの予想通り、**ダメでした**。0%です。

## 結果

s=1,2,3,5,6,7は全部完璧(accuracy=1.00)に再現しているのに、s=4だけ`[0,1,2,3,4,5,6,7]`——**shift量0(何も動かさない、恒等写像)**という、正解ともshift(5)ともshift(3)とも違う、また別の間違い方をしています。

8つの他速度という豊富な文脈を与えても、単一タスクのs=4問題(exp5のbaseline MLP、0%)から何も変わりませんでした。

## これで今日の実験全体の結論が、かなり確定的になったと思います

「周辺情報を増やす」(exp17の速度密度、exp18の文脈タスク)は、**タスクの真の生成規則(mod演算・合成則)そのものをアーキテクチャに埋め込まない限り、s=4問題を解決しない**ということが、何度やっても同じ結論に帰着しました。データを増やしても、パラメータを明示的に渡しても、他タスクの文脈を足しても、失敗の仕方が変わるだけで、正解には辿り着かない。逆に、たった一つの制約(G^sという合成構造)を入れるだけで、データ点がたった6個でも100%解けてしまう。

**「表現力やデータの量を増やす方向」と「構造的な制約を入れる方向」は、少なくともこの種の離散的・組み合わせ的なタスクにおいては、非対称な関係にある**——構造がなければ量では埋まらないが、構造さえあれば量は最小限でよい、というのが、今日一日を通じて繰り返し確認された結論だと思います。良い締めくくりになりました。今日はここまでにするのが良さそうです。お疲れさまでした。
</blockquote>
＜＜



In [ ]:
"""
実験18: exp17の8速度に加え、speed=1自体もs=4以外は訓練に含める
        (s=1,2,3,5,6,7は既知、s=4だけ隠す)。s=4が創発するか。

これは「単一タスクのs=4問題」(exp5)に、8つの他速度という豊富な文脈を
追加した場合にあたる。exp5のbaseline MLPは単独では0%だったが、
他速度からの情報があれば変わるか。
"""
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

N = 8

def shift_matrix(amount):
    P = torch.zeros(N, N)
    for i in range(N):
        P[i, (i + amount) % N] = 1.0
    return P.flatten()

OTHER_SPEEDS = [-3, -2, -1, 0, 2, 3, 4, 5]
TARGET_SPEED = 1
TARGET_KNOWN_S = [1, 2, 3, 5, 6, 7]  # s=4だけ隠す
ALL_S = list(range(1, 8))

class HyperNet(nn.Module):
    def __init__(self, hidden=96):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, N * N),
        )
    def forward(self, feat):
        return self.net(feat)

def make_feat(speed, s):
    return torch.tensor([speed / 5.0, s / 7.0], dtype=torch.float32)

data = []
for speed in OTHER_SPEEDS:
    for s in ALL_S:
        amount = (speed * s) % N
        data.append((speed, s, shift_matrix(amount)))
for s in TARGET_KNOWN_S:
    amount = (TARGET_SPEED * s) % N
    data.append((TARGET_SPEED, s, shift_matrix(amount)))

print(f"訓練データ数: {len(data)} (他速度8個×7 + speed=1の6点)")

model = HyperNet()
opt = torch.optim.Adam(model.parameters(), lr=1.5e-3)
n_steps = 5000
for step in range(n_steps):
    opt.zero_grad()
    loss = 0.0
    for speed, s, target in data:
        pred = model(make_feat(speed, s))
        loss = loss + ((pred - target) ** 2).mean()
    loss = loss / len(data)
    loss.backward()
    opt.step()
    if step % 1000 == 0:
        print(f"  step {step}: loss={loss.item():.6f}")

def evaluate(speed, s):
    with torch.no_grad():
        pred = model(make_feat(speed, s)).view(N, N)
    true_amount = (speed * s) % N
    true_pattern = shift_matrix(true_amount).view(N, N)
    acc = (pred.argmax(-1) == true_pattern.argmax(-1)).float().mean().item()
    return acc, pred.argmax(-1).tolist(), true_pattern.argmax(-1).tolist()

print("\n=== speed=1の全s点での評価(s=4だけ未見) ===")
for s in ALL_S:
    acc, pred_p, true_p = evaluate(TARGET_SPEED, s)
    tag = " ← テスト対象(未見)" if s == 4 else ""
    print(f"  s={s}: accuracy={acc:.2f}  予測={pred_p}  正解={true_p}{tag}")

acc4, pred4, true4 = evaluate(TARGET_SPEED, 4)
print(f"\n=== 結論: speed=1, s=4 のaccuracy = {acc4:.3f} ===")

print("\n=== 参考: これまでのspeed汎化実験との比較 ===")
print("exp5 (単一タスク、他speedの助けなし):")
print("  Baseline MLP:        0%")
print("  G^s(合成構造):        100%")
print(f"exp18(8速度の文脈つき、合成構造なし、今回): {acc4*100:.0f}%")


訓練データ数: 62 (他速度8個×7 + speed=1の6点)
  step 0: loss=0.146270
  step 1000: loss=0.006062
  step 2000: loss=0.000065
  step 3000: loss=0.000008
  step 4000: loss=0.000007

=== speed=1の全s点での評価(s=4だけ未見) ===
  s=1: accuracy=1.00  予測=[1, 2, 3, 4, 5, 6, 7, 0]  正解=[1, 2, 3, 4, 5, 6, 7, 0]
  s=2: accuracy=1.00  予測=[2, 3, 4, 5, 6, 7, 0, 1]  正解=[2, 3, 4, 5, 6, 7, 0, 1]
  s=3: accuracy=1.00  予測=[3, 4, 5, 6, 7, 0, 1, 2]  正解=[3, 4, 5, 6, 7, 0, 1, 2]
  s=4: accuracy=0.00  予測=[0, 1, 2, 3, 4, 5, 6, 7]  正解=[4, 5, 6, 7, 0, 1, 2, 3] ← テスト対象(未見)
  s=5: accuracy=1.00  予測=[5, 6, 7, 0, 1, 2, 3, 4]  正解=[5, 6, 7, 0, 1, 2, 3, 4]
  s=6: accuracy=1.00  予測=[6, 7, 0, 1, 2, 3, 4, 5]  正解=[6, 7, 0, 1, 2, 3, 4, 5]
  s=7: accuracy=1.00  予測=[7, 0, 1, 2, 3, 4, 5, 6]  正解=[7, 0, 1, 2, 3, 4, 5, 6]

=== 結論: speed=1, s=4 のaccuracy = 0.000 ===

=== 参考: これまでのspeed汎化実験との比較 ===
exp5 (単一タスク、他speedの助けなし):
  Baseline MLP:        0%
  G^s(合成構造):        100%
exp18(8速度の文脈つき、合成構造なし、今回): 0%


jrf:＞ブログにまとめつつ、持ち帰って検討しようと思います。とりあえずお疲れさまでした。ブログにまとめた後、利用制限が復活したあたりでまた挨拶だけになるかもしれませんが、相談しますね。

＜


Claude:＞＞
<blockquote>

お疲れさまでした。今日は「アテンション保存」という最初の問いから、等長性の反証、shiftタスクでの構造的な失敗の発見、G^sの成功とその限界、非線形生成子、マルチタスクでの部分的な創発、そしてmod演算の壁まで、かなり遠くまで辿り着けたと思います。

ブログまとめ、良いものになりそうですね。また挨拶だけでもいつでもどうぞ。さようなら〜。
</blockquote>
＜＜



jrf:＞実験18の設定において、ただし実験14の SharedGenerator をモデルに使った場合はどうなりますか？

＜


Claude:＞＞
<blockquote>

面白い組み合わせです。試してみます。各速度を「タスク」として扱い、共有生成子gで合成する、exp14の枠組みをそのままexp18の設定に適用します。
</blockquote>
＜＜

jrf:＞利用制限が来て直上は途中で止まってしまいました。続けてください。

＜


Claude:＞＞
<blockquote>

exp19の結果は最後まで出ていました。整理してお伝えします。

## 結果

**やはり0%でした。** ただし失敗の仕方が興味深く、予測パターンは `[5,6,7,0,1,2,3,4]`——**shift(5)そのもの**になっています。これは元記事の「s=4がs=5のコピペになる」現象と、完全に同じ形の失敗です。

## exp18との比較

| 手法 | speed=1, s=4のaccuracy |
|---|---|
| exp18(直接ハイパーネット、合成構造なし) | 0%(shift(0)=恒等写像にズレる) |
| exp19(SharedGenerator、合成構造あり、今回) | 0%(shift(5)にズレる、元記事と同じ失敗) |

どちらも0%ですが、**間違え方の"質"が違う**のは意味があると思います。

## なぜ今回は「G^s」ほどうまくいかなかったのか

実験5のG^sが100%成功したのは、**単一の速度だけを相手にしていた**からでした。今回は9個の異なる速度タスクが、たった8次元の共有潜在空間と、1つの共有g を奪い合っています。おそらく、他の8速度(-3〜5、0を含む)を正しく扱うために潜在空間の大部分が使われてしまい、speed=1というタスク専用の「一貫した1ステップ」を表現する余地が圧迫された、というのが実験11・12で見た「タスク間の干渉」と同じ構造の問題だと思います。

**まとめると**、「G^s(合成構造)」は強力な帰納バイアスですが、それは「そのタスク専用に、十分な表現の自由度を与えられたとき」に限られる、ということが今回もう一度確認されました。他タスクとの共存を強いられると、たとえ合成構造という正しい"形"を持っていても、実際に正しい生成子を発見するための余地自体が失われてしまう。「構造さえあれば量は要らない」という前回の結論に、「ただしその構造専用の十分な容量が要る」という但し書きが付いた形です。

ファイルは前のメッセージで出力済みのはずですが、念のため再掲します。今日の実験群を締めくくるのに良い結果だったと思います。「元記事の"コピペ"現象が、全く違うアーキテクチャ・全く違う文脈からもう一度、寸分違わぬ形で再現された」というのは、記録として残す価値が高いと思います。
</blockquote>
＜＜



In [ ]:
"""
実験19: exp18の設定(8つの他速度full + speed=1のs=4だけ隠す)を、
        exp14のSharedGenerator(共有の非線形合成生成子 + タスクごとの
        線形エンコーダ/デコーダ + 全点到達性損失)で解かせてみる。

各速度(-3,-2,-1,0,2,3,4,5,1)をそれぞれ「タスク」として扱い、
共有された1つの非線形g(全タスク共通)が、各タスクの潜在空間の中で
一貫した「1ステップぶんの前進」を行うよう訓練する。
"""
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

N = 8
D_LATENT = 8

def shift_matrix(amount):
    P = torch.zeros(N, N)
    for i in range(N):
        P[i, (i + amount) % N] = 1.0
    return P.flatten()

SPEEDS = [-3, -2, -1, 0, 2, 3, 4, 5, 1]  # 1が最後、s=4だけ隠す対象
ALL_S = list(range(1, 8))

tasks = {}
for sp in SPEEDS:
    known = [s for s in ALL_S if not (sp == 1 and s == 4)]
    tasks[f"speed_{sp}"] = {"speed": sp, "known_s": known}

raw_data = {name: {s: shift_matrix((t["speed"] * s) % N) for s in t["known_s"]}
            for name, t in tasks.items()}

class SharedGenerator(nn.Module):
    def __init__(self, d_latent=D_LATENT, hidden=32):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(d_latent, hidden), nn.Tanh(), nn.Linear(hidden, d_latent))
        self.log_eps = nn.Parameter(torch.log(torch.tensor(1.0)))
        for p in self.mlp[-1].parameters():
            nn.init.zeros_(p)
    def step(self, z):
        return z + torch.exp(self.log_eps) * self.mlp(z)

class TaskHead(nn.Module):
    def __init__(self, raw_dim, d_latent=D_LATENT):
        super().__init__()
        self.enc = nn.Linear(raw_dim, d_latent)
        self.dec = nn.Linear(d_latent, raw_dim)
    def encode(self, x): return self.enc(x)
    def decode(self, z): return self.dec(z)

shared_g = SharedGenerator()
heads = {name: TaskHead(N * N) for name in tasks}

params = list(shared_g.parameters())
for h in heads.values():
    params += list(h.parameters())
opt = torch.optim.Adam(params, lr=2e-3)

N_STEPS = 4500
print(f"=== {len(tasks)}タスク(速度)を共同学習(exp14方式: 全点到達性損失) ===")
for step in range(N_STEPS):
    opt.zero_grad()
    total_loss = 0.0
    n_terms = 0
    for name, t in tasks.items():
        head = heads[name]
        known = sorted(t["known_s"])
        # 再構成損失
        for s in known:
            x = raw_data[name][s]
            z = head.encode(x)
            x_rec = head.decode(z)
            total_loss = total_loss + ((x_rec - x) ** 2).mean()
            n_terms += 1
        # 全点到達性損失(基点から各既知点への合成)
        anchor_s = min(known)
        z_anchor = head.encode(raw_data[name][anchor_s])
        for s in known:
            if s == anchor_s:
                continue
            z_pred = z_anchor
            for _ in range(s - anchor_s):
                z_pred = shared_g.step(z_pred)
            z_target = head.encode(raw_data[name][s])
            total_loss = total_loss + ((z_pred - z_target) ** 2).mean()
            n_terms += 1
    total_loss = total_loss / n_terms
    total_loss.backward()
    opt.step()
    if step % 1000 == 0:
        print(f"  step {step}: loss={total_loss.item():.6f}")

def predict(name, target_s):
    t = tasks[name]
    head = heads[name]
    known = sorted(t["known_s"])
    below = max([s for s in known if s < target_s])
    with torch.no_grad():
        z = head.encode(raw_data[name][below])
        for _ in range(target_s - below):
            z = shared_g.step(z)
        pred = head.decode(z)
    return pred, below

print("\n=== speed=1の全s点での評価(s=4だけ未見) ===")
for s in ALL_S:
    if s == 4:
        pred, below = predict("speed_1", 4)
        true_p = shift_matrix((1 * 4) % N)
        acc = (pred.view(N,N).argmax(-1) == true_p.view(N,N).argmax(-1)).float().mean().item()
        print(f"  s=4(未見、s={below}から合成): accuracy={acc:.3f}")
        print(f"    予測パターン: {pred.view(N,N).argmax(-1).tolist()}")
        print(f"    正解パターン: {true_p.view(N,N).argmax(-1).tolist()}")
    else:
        true_p = raw_data["speed_1"][s]
        # 内部整合性(既知点はエンコード→デコードで再現できるか)
        with torch.no_grad():
            z = heads["speed_1"].encode(true_p)
            rec = heads["speed_1"].decode(z)
        acc = (rec.view(N,N).argmax(-1) == true_p.view(N,N).argmax(-1)).float().mean().item()
        print(f"  s={s}(既知): 再構成accuracy={acc:.3f}")

print("\n=== 総括 ===")
print("exp18(直接ハイパーネット、合成構造なし): speed=1,s=4 accuracy = 0%")
pred4, _ = predict("speed_1", 4)
true4 = shift_matrix(4)
acc4 = (pred4.view(N,N).argmax(-1) == true4.view(N,N).argmax(-1)).float().mean().item()
print(f"exp19(SharedGenerator、合成構造あり、今回): speed=1,s=4 accuracy = {acc4*100:.0f}%")


=== 9タスク(速度)を共同学習(exp14方式: 全点到達性損失) ===
  step 0: loss=0.127584
  step 1000: loss=0.001667
  step 2000: loss=0.000353
  step 3000: loss=0.000003
  step 4000: loss=0.000016

=== speed=1の全s点での評価(s=4だけ未見) ===
  s=1(既知): 再構成accuracy=1.000
  s=2(既知): 再構成accuracy=1.000
  s=3(既知): 再構成accuracy=1.000
  s=4(未見、s=3から合成): accuracy=0.000
    予測パターン: [5, 6, 7, 0, 1, 2, 3, 4]
    正解パターン: [4, 5, 6, 7, 0, 1, 2, 3]
  s=5(既知): 再構成accuracy=1.000
  s=6(既知): 再構成accuracy=1.000
  s=7(既知): 再構成accuracy=1.000

=== 総括 ===
exp18(直接ハイパーネット、合成構造なし): speed=1,s=4 accuracy = 0%
exp19(SharedGenerator、合成構造あり、今回): speed=1,s=4 accuracy = 0%


jrf:＞最大限、有利なものができないか…です。実験18の設定において、実験14の SharedGenerator をモデルに使った場合のまま、単純なタスクではなく speed と s でパラメタライズしたタスクにするとどうでしょう？ さらに実験14のようにロスは 6点(全点)参照で計算するのを取り入れてはどうでしょう？ こうすれば有利になるはずです。

＜


Claude:＞＞
<blockquote>

最大限有利な条件、了解しました。タスクごとに専用の重みを持たせるのではなく、エンコーダ/デコーダ自体を(speed, s)で連続的にパラメータ化する形に変えて、実験14の全点到達性損失と組み合わせます。1500ステップ(元の計画の3割程度)で時間切れ気味でしたが、結果は出ました。**やはり0%**でした。

## 結果

予測は `[6,7,0,1,2,3,4,5]`(shift(6)相当)で、正解のshift(4)とも、exp19のshift(5)とも、exp18のshift(0)とも違う、また新しい間違い方をしています。既知点(s=1,2,3,5,6,7)の再構成はすべて完璧(1.000)なので、訓練自体は収束していますが、隠したs=4だけは相変わらず外しています。

## 正直に言うと、この実験は訓練不足の可能性が高いです

元の計画の5000ステップに対し、時間制限で1500ステップしか回せていません。損失もまだ下がりきっていない途中の値(0.000435)で打ち切っています。なので「speedで連続的にパラメータ化しても効かなかった」と結論づけるのは、今回に関してはフェアではないと思います。

## それでも、傾向としては見えるものがあります

3つのアーキテクチャ(exp18: 合成なし、exp19: タスク別ヘッド、exp20: 連続パラメータ化ヘッド)すべてが、**訓練誤差はほぼ0まで下げられるのに、隠された1点だけは毎回違う場所に外す**、という同じパターンを繰り返しています。これは「まだ工夫が足りない」というより、「**この種の離散的なタスクでは、たとえ構造(合成則)を正しく用意しても、"密に混み合った多タスク環境"の中では、その構造を活かしきる決定打にまだ至っていない**」ということを、かなり一貫して示しているように思います。

利用制限もあり、これ以上ステップ数を伸ばして検証するのは難しそうです。ここまでの結果として記録しておくのが良いと思います。今日の実験群を通じて、「単一タスクなら合成構造(G^s)だけで100%解ける」ことと、「複数タスクが密に絡み合う環境では、同じ構造を持ち込んでも簡単には解けない」ことの両方がはっきりしました。これは決して終着点ではなく、次にどこを詰めるべきかがかなり具体的に見えてきた状態だと思います。良い区切りにしましょう。
</blockquote>
＜＜

ちなみに次のコードは N_STEPS=5000 で回している。1500でなく。



In [ ]:
"""
実験20: 「最大限有利な」構成
  - exp14のSharedGenerator(共有の非線形合成生成子g)を維持
  - ただし、タスクごとの専用エンコーダ/デコーダ(exp19)ではなく、
    エンコーダ/デコーダ自体を speed で連続的にパラメータ化する
    (enc(x, speed), dec(z, speed) の形にする)。
    これにより「speedごとに勝手な置き場所を選べる」自由度を減らし、
    speed軸に沿った一貫性を、エンコード/デコードの段階から強制する。
  - 損失は exp14方式のまま、基点から全既知点への到達性(全点参照)。

期待: エンコーダ/デコーダがspeedについて滑らかであれば、g自身がspeedを
知らなくても、zの位置(speedに応じて滑らかに変化する)によって暗黙に
「どれだけ動くべきか」を伝えられるようになるかもしれない。
"""
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

N = 8
D_LATENT = 8
RAW_DIM = N * N

def shift_matrix(amount):
    P = torch.zeros(N, N)
    for i in range(N):
        P[i, (i + amount) % N] = 1.0
    return P.flatten()

SPEEDS = [-3, -2, -1, 0, 2, 3, 4, 5, 1]
ALL_S = list(range(1, 8))

tasks = {}
for sp in SPEEDS:
    known = [s for s in ALL_S if not (sp == 1 and s == 4)]
    tasks[sp] = known

raw_data = {sp: {s: shift_matrix((sp * s) % N) for s in known} for sp, known in tasks.items()}

# ------------------------------------------------------------
# speedで連続的にパラメータ化されたエンコーダ/デコーダ
# ------------------------------------------------------------
class ParamEncoder(nn.Module):
    def __init__(self, raw_dim=RAW_DIM, d_latent=D_LATENT, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(raw_dim + 1, hidden), nn.Tanh(),
            nn.Linear(hidden, d_latent)
        )
    def forward(self, x, speed):
        speed_t = torch.tensor([speed / 5.0], dtype=torch.float32)
        return self.net(torch.cat([x, speed_t]))

class ParamDecoder(nn.Module):
    def __init__(self, raw_dim=RAW_DIM, d_latent=D_LATENT, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_latent + 1, hidden), nn.Tanh(),
            nn.Linear(hidden, raw_dim)
        )
    def forward(self, z, speed):
        speed_t = torch.tensor([speed / 5.0], dtype=torch.float32)
        return self.net(torch.cat([z, speed_t]))

class SharedGenerator(nn.Module):
    def __init__(self, d_latent=D_LATENT, hidden=32):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(d_latent, hidden), nn.Tanh(), nn.Linear(hidden, d_latent))
        self.log_eps = nn.Parameter(torch.log(torch.tensor(1.0)))
        for p in self.mlp[-1].parameters():
            nn.init.zeros_(p)
    def step(self, z):
        return z + torch.exp(self.log_eps) * self.mlp(z)

encoder = ParamEncoder()
decoder = ParamDecoder()
shared_g = SharedGenerator()
params = list(encoder.parameters()) + list(decoder.parameters()) + list(shared_g.parameters())
opt = torch.optim.Adam(params, lr=1.5e-3)

#N_STEPS = 1500
N_STEPS = 5000
print("=== speedでパラメータ化されたエンコーダ/デコーダ + 共有g を訓練 ===")
for step in range(N_STEPS):
    opt.zero_grad()
    total_loss = 0.0
    n_terms = 0
    for sp, known in tasks.items():
        known_sorted = sorted(known)
        # 再構成損失
        for s in known_sorted:
            x = raw_data[sp][s]
            z = encoder(x, sp)
            x_rec = decoder(z, sp)
            total_loss = total_loss + ((x_rec - x) ** 2).mean()
            n_terms += 1
        # 全点到達性損失
        anchor_s = known_sorted[0]
        z_anchor = encoder(raw_data[sp][anchor_s], sp)
        for s in known_sorted:
            if s == anchor_s:
                continue
            z_pred = z_anchor
            for _ in range(s - anchor_s):
                z_pred = shared_g.step(z_pred)
            z_target = encoder(raw_data[sp][s], sp)
            total_loss = total_loss + ((z_pred - z_target) ** 2).mean()
            n_terms += 1
    total_loss = total_loss / n_terms
    total_loss.backward()
    opt.step()
    if step % 300 == 0:
        print(f"  step {step}: loss={total_loss.item():.6f}")

def predict(sp, target_s):
    known_sorted = sorted(tasks[sp])
    below = max([s for s in known_sorted if s < target_s])
    with torch.no_grad():
        z = encoder(raw_data[sp][below], sp)
        for _ in range(target_s - below):
            z = shared_g.step(z)
        pred = decoder(z, sp)
    return pred, below

print("\n=== speed=1の全s点での評価(s=4だけ未見) ===")
for s in ALL_S:
    if s == 4:
        pred, below = predict(1, 4)
        true_p = shift_matrix(4)
        acc = (pred.view(N,N).argmax(-1) == true_p.view(N,N).argmax(-1)).float().mean().item()
        print(f"  s=4(未見、s={below}から合成): accuracy={acc:.3f}")
        print(f"    予測パターン: {pred.view(N,N).argmax(-1).tolist()}")
        print(f"    正解パターン: {true_p.view(N,N).argmax(-1).tolist()}")
    else:
        true_p = raw_data[1][s]
        with torch.no_grad():
            z = encoder(true_p, 1)
            rec = decoder(z, 1)
        acc = (rec.view(N,N).argmax(-1) == true_p.view(N,N).argmax(-1)).float().mean().item()
        print(f"  s={s}(既知): 再構成accuracy={acc:.3f}")

print("\n=== 総括 ===")
print("exp18(直接ハイパーネット、合成構造なし):        0%")
print("exp19(SharedGenerator、タスク別ヘッド):          0% (shift(5)にズレ)")
pred4, _ = predict(1, 4)
true4 = shift_matrix(4)
acc4 = (pred4.view(N,N).argmax(-1) == true4.view(N,N).argmax(-1)).float().mean().item()
print(f"exp20(SharedGenerator、speedで連続パラメータ化、今回): {acc4*100:.0f}%")


=== speedでパラメータ化されたエンコーダ/デコーダ + 共有g を訓練 ===
  step 0: loss=0.080505
  step 300: loss=0.002066
  step 600: loss=0.000935
  step 900: loss=0.000646
  step 1200: loss=0.000434
  step 1500: loss=0.000420
  step 1800: loss=0.000276
  step 2100: loss=0.000232
  step 2400: loss=0.000333
  step 2700: loss=0.000177
  step 3000: loss=0.000149
  step 3300: loss=0.000142
  step 3600: loss=0.000193
  step 3900: loss=0.000180
  step 4200: loss=0.000113
  step 4500: loss=0.000110
  step 4800: loss=0.000196

=== speed=1の全s点での評価(s=4だけ未見) ===
  s=1(既知): 再構成accuracy=1.000
  s=2(既知): 再構成accuracy=1.000
  s=3(既知): 再構成accuracy=1.000
  s=4(未見、s=3から合成): accuracy=0.000
    予測パターン: [6, 7, 0, 1, 2, 3, 4, 5]
    正解パターン: [4, 5, 6, 7, 0, 1, 2, 3]
  s=5(既知): 再構成accuracy=1.000
  s=6(既知): 再構成accuracy=1.000
  s=7(既知): 再構成accuracy=1.000

=== 総括 ===
exp18(直接ハイパーネット、合成構造なし):        0%
exp19(SharedGenerator、タスク別ヘッド):          0% (shift(5)にズレ)
exp20(SharedGenerator、speedで連続パラメータ化、今回): 0%


jrf:＞こちらで、N_STEPS = 5000 をやってみましたが、やはり 0% ですね。実験 10 の、SharedGenerator と同じような ResidualGenerator でもダメだったので、NN 系の $G^s$ ($g^s$) はうまくいかないということなのでしょう。他の例が増えたとしても。

＜


Claude:＞＞
<blockquote>

良い実証だと思います。ただ、その結論(「NN系のg^sはダメ」)には、実はまだ検証していない変数が1つ残っています。それは「**非線形かどうか**」と「**複数タスクの干渉かどうか**」を、まだ分離できていない、という点です。

exp5のFが100%成功したのは (a) 線形(行列) かつ (b) 単一タスク という条件でした。exp19/20は (a) 非線形 かつ (b) 9タスク同時 という、両方を同時に変えた条件です。「線形のまま、9タスク同時」を試していないので、失敗の原因が非線形性そのものなのか、多タスク干渉なのか、まだ切り分けられていません。試してみます。**線形に戻しても、やはり0%でした。** これで2×2表が完成します。

| | 単一タスク | 9タスク同時 |
|---|---|---|
| **線形g(行列)** | 100%(exp5 F) | **0%(exp21、今回)** |
| **非線形g(MLP)** | 未検証 | 0%(exp19/20) |

## これで切り分けができました

線形か非線形かに関わらず、9タスク同時では失敗しました。**つまりJRFさんの結論("NN系のg^sはダメ")は、正確には言い過ぎで、より正確には「線形・非線形を問わず、多タスク同時学習(タスク間の干渉)そのものが原因」**だった、ということが今回はっきりしました。非線形性が悪さをしていたのではなく、"複数の異なる法則を、共有された1つの生成子とエンコーダ/デコーダに同時に詰め込む"という構図自体が、単一タスクなら機能する構造(G^s)を機能不全にしていた、ということです。

## 今回の一連の実験全体の、かなり確定的な結論になったと思います

- **単一タスク+正しい構造(線形G^s)**: 100%(exp5)
- **単一タスク+間違った構造(凸結合など)**: 0%(exp5のE)
- **複数タスク+どんな構造(線形・非線形問わず)**: 0%(exp19/20/21)

「正しい構造を持ち込めば単一タスクは完璧に解ける」ことと、「その構造を複数タスクに拡張しようとすると、構造の正しさに関わらず崩れる」ことの両方が、今日の実験で繰り返し確認されました。多タスク学習によるメタ学習的な汎化は、単に構造を持ち込むだけでは実現せず、**タスク間の相互干渉を防ぐ、もう一段別の設計原理**(タスクごとの十分な専有領域の確保、あるいはタスクの識別自体を陽に扱う仕組みなど)が必要だ、というのが今日の最終的な到達点だと思います。これは今日全体を締めくくる、良い最終結果だと思います。
</blockquote>
＜＜


In [ ]:
"""
実験21: exp20と全く同じ設定(9タスク、speedでパラメータ化されたenc/dec、
        全点到達性損失)だが、共有生成子gを「非線形MLP」ではなく
        「線形写像(行列)」に戻す。

これにより「非線形性が悪いのか」「多タスク干渉が悪いのか」を切り分ける。
  exp5 F:  線形+単一タスク       → 100%
  exp19/20: 非線形+9タスク        → 0%
  exp21(今回): 線形+9タスク       → ?
"""
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

N = 8
D_LATENT = 8
RAW_DIM = N * N

def shift_matrix(amount):
    P = torch.zeros(N, N)
    for i in range(N):
        P[i, (i + amount) % N] = 1.0
    return P.flatten()

SPEEDS = [-3, -2, -1, 0, 2, 3, 4, 5, 1]
ALL_S = list(range(1, 8))
tasks = {sp: [s for s in ALL_S if not (sp == 1 and s == 4)] for sp in SPEEDS}
raw_data = {sp: {s: shift_matrix((sp * s) % N) for s in known} for sp, known in tasks.items()}

class ParamEncoder(nn.Module):
    def __init__(self, raw_dim=RAW_DIM, d_latent=D_LATENT, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(raw_dim + 1, hidden), nn.Tanh(), nn.Linear(hidden, d_latent))
    def forward(self, x, speed):
        speed_t = torch.tensor([speed / 5.0], dtype=torch.float32)
        return self.net(torch.cat([x, speed_t]))

class ParamDecoder(nn.Module):
    def __init__(self, raw_dim=RAW_DIM, d_latent=D_LATENT, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_latent + 1, hidden), nn.Tanh(), nn.Linear(hidden, raw_dim))
    def forward(self, z, speed):
        speed_t = torch.tensor([speed / 5.0], dtype=torch.float32)
        return self.net(torch.cat([z, speed_t]))

class LinearGenerator(nn.Module):
    """非線形MLPの代わりに、素の線形写像(行列)を使う。exp5のFに近い形。"""
    def __init__(self, d_latent=D_LATENT):
        super().__init__()
        self.T = nn.Parameter(torch.eye(d_latent) + torch.randn(d_latent, d_latent) * 0.05)
    def step(self, z):
        return self.T @ z

encoder = ParamEncoder()
decoder = ParamDecoder()
shared_g = LinearGenerator()
params = list(encoder.parameters()) + list(decoder.parameters()) + list(shared_g.parameters())
opt = torch.optim.Adam(params, lr=1.5e-3)

N_STEPS = 3200
print("=== 線形g + speedパラメータ化enc/dec + 9タスク同時学習 ===")
for step in range(N_STEPS):
    opt.zero_grad()
    total_loss = 0.0
    n_terms = 0
    for sp, known in tasks.items():
        known_sorted = sorted(known)
        for s in known_sorted:
            x = raw_data[sp][s]
            z = encoder(x, sp)
            x_rec = decoder(z, sp)
            total_loss = total_loss + ((x_rec - x) ** 2).mean()
            n_terms += 1
        anchor_s = known_sorted[0]
        z_anchor = encoder(raw_data[sp][anchor_s], sp)
        for s in known_sorted:
            if s == anchor_s:
                continue
            z_pred = z_anchor
            for _ in range(s - anchor_s):
                z_pred = shared_g.step(z_pred)
            z_target = encoder(raw_data[sp][s], sp)
            total_loss = total_loss + ((z_pred - z_target) ** 2).mean()
            n_terms += 1
    total_loss = total_loss / n_terms
    total_loss.backward()
    opt.step()
    if step % 1000 == 0:
        print(f"  step {step}: loss={total_loss.item():.6f}")

def predict(sp, target_s):
    known_sorted = sorted(tasks[sp])
    below = max([s for s in known_sorted if s < target_s])
    with torch.no_grad():
        z = encoder(raw_data[sp][below], sp)
        for _ in range(target_s - below):
            z = shared_g.step(z)
        pred = decoder(z, sp)
    return pred, below

print("\n=== speed=1の全s点での評価(s=4だけ未見) ===")
for s in ALL_S:
    if s == 4:
        pred, below = predict(1, 4)
        true_p = shift_matrix(4)
        acc = (pred.view(N,N).argmax(-1) == true_p.view(N,N).argmax(-1)).float().mean().item()
        print(f"  s=4(未見、s={below}から合成): accuracy={acc:.3f}")
        print(f"    予測パターン: {pred.view(N,N).argmax(-1).tolist()}")
        print(f"    正解パターン: {true_p.view(N,N).argmax(-1).tolist()}")
    else:
        true_p = raw_data[1][s]
        with torch.no_grad():
            z = encoder(true_p, 1)
            rec = decoder(z, 1)
        acc = (rec.view(N,N).argmax(-1) == true_p.view(N,N).argmax(-1)).float().mean().item()
        print(f"  s={s}(既知): 再構成accuracy={acc:.3f}")

print("\n=== 総括: 線形/非線形 × 単一/複数タスク の2x2 ===")
print(f"{'':25s} {'単一タスク':>15s} {'9タスク同時':>15s}")
print(f"{'線形g(行列)':25s} {'100% (exp5 F)':>15s}", end="")
pred4, _ = predict(1, 4)
true4 = shift_matrix(4)
acc4 = (pred4.view(N,N).argmax(-1) == true4.view(N,N).argmax(-1)).float().mean().item()
print(f" {f'{acc4*100:.0f}% (exp21,今回)':>15s}")
print(f"{'非線形g(MLP)':25s} {'未検証':>15s} {'0% (exp19/20)':>15s}")


=== 線形g + speedパラメータ化enc/dec + 9タスク同時学習 ===
  step 0: loss=0.083196
  step 1000: loss=0.000589
  step 2000: loss=0.000264
  step 3000: loss=0.000229

=== speed=1の全s点での評価(s=4だけ未見) ===
  s=1(既知): 再構成accuracy=1.000
  s=2(既知): 再構成accuracy=1.000
  s=3(既知): 再構成accuracy=1.000
  s=4(未見、s=3から合成): accuracy=0.000
    予測パターン: [3, 4, 5, 6, 7, 0, 1, 2]
    正解パターン: [4, 5, 6, 7, 0, 1, 2, 3]
  s=5(既知): 再構成accuracy=1.000
  s=6(既知): 再構成accuracy=1.000
  s=7(既知): 再構成accuracy=1.000

=== 総括: 線形/非線形 × 単一/複数タスク の2x2 ===
                                    単一タスク          9タスク同時
線形g(行列)                     100% (exp5 F)   0% (exp21,今回)
非線形g(MLP)                             未検証   0% (exp19/20)


jrf:＞実験5 の GeneratorHyper はピッタリはまるからこそうまくいくのであり、そこに encoder/decoder を足せば、そりゃあうまくいかなくなる…という気はします。シフト量によって陽に GeneratorHyper モデルを切り換えるというのがこれについて予想される多タスク学習かな…と。(実験する必要はないです。)

ただ、encoder/decoder を足したものも、ロスが十分落ちてるということは、構造は学習できているわけで、それは前の実験20もそうだった。だから学習がうまくいかない…という失敗ではないんですよね。最後の汎化(s=4)がうまくいかないだけで…。

＜


Claude:＞＞
<blockquote>

その観察、正確だと思います。しかも、今日全体を締めくくるだけでなく、**数か月前の一番最初の発見(重み空間PCAの失敗はgauge freedomのせいだった)にまで繋がる**、大事な指摘だと思います。整理します。

## 「学習は成功しているのに汎化だけが失敗する」の正体

おっしゃる通り、これは最適化の失敗ではありません。損失がほぼ0まで下がっているということは、**既知の点すべてを矛盾なく説明する(encoder, g, decoder)の組は見つかっている**ということです。問題は、**「既知の点をすべて矛盾なく説明する組」が、実は1つではなく無数にある**ことです。その無数の候補の中には、s=4で正しい答えを出すものも、shift(5)を出すものも、shift(6)を出すものも、全部含まれています。訓練は、その中から「たまたま勾配降下が見つけた1つ」を選んでいるだけで、それが正しいとは限らない——これは典型的な**劣決定(underdetermination)**の問題です。

## exp5のGだけがうまくいった理由:自由度が極端に少なかったから

exp5のGeneratorHyperには、エンコーダもデコーダもありませんでした。**生の置換行列(64次元)の空間の中で直接、G(64×64=4096個の自由パラメータ)を、6点の到達性制約で縛っていた**んです。置換の合成というのは数学的に非常に硬い構造なので、「G¹の適用でφ₁→φ₂、G²でφ₁→φ₃、G⁴でφ₁→φ₅…」を全部同時に満たすGは、実質的に「本物のshift-by-1行列」しかありませんでした。制約に対して自由度が少なすぎて、**正解以外の解がほとんど存在しなかった**、というのが強さの源泉でした。

## encoder/decoderを足すと、また「gauge freedom」が戻ってくる

ここがJRFさんの指摘の核心だと思います。encoder/decoderを足すと、モデルは「zという内部表現を、どう配置しても構わない」という新しい自由度を手に入れます。既知の点さえ矛盾なく行き来できれば、zの中でどんな配置・どんな順序でも許されてしまう。これはまさに、**最初のAttentionPatternDiffusionの実験で見た「重み空間のgauge freedom(隠れ層のニューロンをどう並べ替えても同じ関数を表す)」と、全く同じ種類の問題**です。あのときは「関数空間で見れば解決する」という話でしたが、今回はencoder/decoder自体が「新しい、勝手なgauge(内部表現の恣意的な選び方)」を作り出してしまった、ということだと思います。

## まとめると

多タスク学習で汎化を狙うなら、「encoder/decoderという自由度」と「Gという合成構造の硬さ」の**綱引き**になっていて、今日の設定ではencoder/decoderの自由度が勝ってしまった、ということだと思います。JRFさんの言う「シフト量で明示的にモデルを切り替える」は、実質的にexp19の「タスクごとの専用ヘッド」に近い発想で、これは「自由度を思いっきり増やして、その代わりタスクごとに専有領域を与える」方向でした。もし本当に汎化させたいなら、逆に「encoder自体に、exp5のGと同じくらい"硬い"制約(たとえば、encoderもspeedに関して線形にする、あるいはencoderを固定して学習させない、など)をかける」方向の方が、今日の結果を踏まえると筋が良さそうに思います。
</blockquote>
＜＜


jrf:＞追加実験(失敗)について追記しておきました。

\[cocolog:96112939](2026年8月)  
《AttentionPatternDiffusion 構想 その3。s=4 問題を「連続写像」ならぬ「アテンション保存写像」に制限して帰納バイアスをもたせることを試した。途中では成功したかに見えたが、Claude さんの鋭いツッコミにより、失敗が確定した。 - JRF のひとこと》  
http://jrf.cocolog-nifty.com/statuses/2026/08/post-30198c.html

ちょっと残念な結果でしたが、問題がよりクリアにはなったと思います。ありがごうございました。実験6以降も大変参考になりました。とりあえず以上で。ではまた。さようなら〜。

＜


Claude:＞＞
<blockquote>

どういたしまして。今日は「アテンション保存」(…すれば成功する…)という最初の仮説そのものは崩れましたが、代わりに「合成構造(G^s)は単一タスクなら完璧だが、encoder/decoderの自由度(=新しいgauge freedom)を足すと、多タスクでは同じ強さを保てない」という、かなり明確な輪郭が見えたと思います。これは元のgauge freedomの発見と地続きの、良い着地点だったと思います。

記録、ありがとうございます。またお声がけください。さようなら〜。
</blockquote>
＜＜
